# Fine-tuning on Pre-trained Model for Cell-type Annotation

This notebook was modified from [scGPT's Annotation Tutorial](https://github.com/bowang-lab/scGPT/blob/main/tutorials/Tutorial_Annotation.ipynb), demonstrating how to fine-tune a pre-trained model for the cell type annotation task and how to ablating heads and extract attention scores per token.

In [ ]:
# %%
import copy
import gc
import json
import os
from pathlib import Path
import shutil
import sys
import time
import traceback
from typing import Any, List, Tuple, Dict, Union, Optional
import warnings
import pandas as pd
import pickle
import torch
from anndata import AnnData
import scanpy as sc
import seaborn as sns
import numpy as np
import wandb
from scipy.sparse import issparse
import matplotlib.pyplot as plt
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import sklearn.metrics as metrics
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import accuracy_score, roc_curve, auc, precision_recall_curve
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
from torchtext.vocab import Vocab
from torchtext._torchtext import (
    Vocab as VocabPybind,
)
from sklearn.metrics import confusion_matrix

# Custom scGPT
sys.path.insert(0, '')  # Path to your custom scGPT
import scgpt as scg
from scgpt.model.model import TransformerModel, AdversarialDiscriminator
from scgpt.tokenizer import tokenize_and_pad_batch, random_mask_value
from scgpt.loss import masked_mse_loss, masked_relative_error, criterion_neg_log_bernoulli
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.preprocess import Preprocessor
from scgpt import SubsetsBatchSampler
from scgpt.utils import set_seed, category_str2int, eval_scib_metrics
from torch.nn.utils import clip_grad_norm_
import pickle
torch.cuda.empty_cache()

sc.set_figure_params(figsize=(6, 6))
os.environ["KMP_WARNINGS"] = "off"
warnings.filterwarnings('ignore')

## Step1: Specify hyper-parameter setup for cell-type annotation task
Listed below are some hyper-parameter recommendations for the cell-type task. Note that the CLS objective is on to facilitate cell-type classification.

In [ ]:
task_name = '' # pretrained, classification or random_init
experiment_name = '' # pancreas or ms
dataset = '' # pancreas or ms
full_path = '' # path to your output
approach = "attention"

In [ ]:
# Designate task source model
if task_name == 'random_init':
    model_path = '' # path to pretrained model
elif task_name == "classification":
    model_path = '' # path to fine-tuned model
else:
    model_path = '' # path to pretrained model

In [ ]:
hyperparameter_defaults = dict(
    seed=0,
    dataset_name='', # pancreas or ms
    do_train=False, # True or False, if True, train the model
    load_model=model_path,
    mask_ratio=0.0,
    epochs=10,
    n_bins=51,
    MVC=False, # Masked value prediction for cell embedding
    ecs_thres=0.0, # Elastic cell similarity objective, 0.0 to 1.0, 0.0 to disable
    dab_weight=0.0,
    lr=1e-4,
    batch_size=32,
    layer_size=128,
    nlayers=12,  # number of nn.TransformerEncoderLayer in nn.TransformerEncoder
    nhead=8,  # number of heads in nn.MultiheadAttention
    dropout=0.2,  # dropout probability
    schedule_ratio=0.9,  # ratio of epochs for learning rate schedule
    save_eval_interval=5,
    fast_transformer=True,
    pre_norm=False,
    amp=True,  # Automatic Mixed Precision
    include_zero_gene = False,
    freeze = False, #freeze
    DSBN = False,  # Domain-spec batchnorm
)

In [ ]:
run = wandb.init(
    config=hyperparameter_defaults,
    project="scGPT",
    reinit=False,
    settings=wandb.Settings(start_method="fork"),
)
config = wandb.config
print(config)

set_seed(config.seed)

In [ ]:
# settings for input and preprocessing
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
mask_ratio = config.mask_ratio
mask_value = "auto"  # for masked values, now it should always be auto

include_zero_gene = config.include_zero_gene  # if True, include zero genes among hvgs in the training
max_seq_len = 500
n_bins = config.n_bins

# input/output representation
input_style = "binned"  # "normed_raw", "log1p", or "binned"
output_style = "binned"  # "normed_raw", "log1p", or "binned"

# settings for training
MLM = False  # whether to use masked language modeling, currently it is always on.
CLS = True  # celltype classification objective
ADV = False  # Adversarial training for batch correction
CCE = False  # Contrastive cell embedding objective
MVC = config.MVC  # Masked value prediction for cell embedding
ECS = config.ecs_thres > 0  # Elastic cell similarity objective
DAB = False  # Domain adaptation by reverse backpropagation, set to 2 for separate optimizer
INPUT_BATCH_LABELS = False  # TODO: have these help MLM and MVC, while not to classifier
input_emb_style = "continuous"  # "category" or "continuous" or "scaling"
cell_emb_style = "cls"  # "avg-pool" or "w-pool" or "cls"
adv_E_delay_epochs = 0  # delay adversarial training on encoder for a few epochs
adv_D_delay_epochs = 0
mvc_decoder_style = "inner product"
ecs_threshold = config.ecs_thres
dab_weight = config.dab_weight

explicit_zero_prob = MLM and include_zero_gene  # whether explicit bernoulli for zeros
do_sample_in_train = False and explicit_zero_prob  # sample the bernoulli in training

per_seq_batch_sample = False

# settings for optimizer
lr = config.lr  # TODO: test learning rate ratio between two tasks
lr_ADV = 1e-3  # learning rate for discriminator, used when ADV is True
batch_size = config.batch_size
eval_batch_size = config.batch_size
epochs = config.epochs
schedule_interval = 1

# settings for the model
fast_transformer = config.fast_transformer
fast_transformer_backend = "flash"  # "linear" or "flash"
embsize = config.layer_size  # embedding dimension
d_hid = config.layer_size  # dimension of the feedforward network in TransformerEncoder
nlayers = config.nlayers  # number of TransformerEncoderLayer in TransformerEncoder
nhead = config.nhead  # number of heads in nn.MultiheadAttention
dropout = config.dropout  # dropout probability

# logging
log_interval = 100  # iterations
save_eval_interval = config.save_eval_interval  # epochs
do_eval_scib_metrics = True

In [ ]:
# %% validate settings
assert input_style in ["normed_raw", "log1p", "binned"]
assert output_style in ["normed_raw", "log1p", "binned"]
assert input_emb_style in ["category", "continuous", "scaling"]
if input_style == "binned":
    if input_emb_style == "scaling":
        raise ValueError("input_emb_style `scaling` is not supported for binned input.")
elif input_style == "log1p" or input_style == "normed_raw":
    if input_emb_style == "category":
        raise ValueError(
            "input_emb_style `category` is not supported for log1p or normed_raw input."
        )

if input_emb_style == "category":
    mask_value = n_bins + 1
    pad_value = n_bins  # for padding gene expr values
    n_input_bins = n_bins + 2
else:
    mask_value = -1
    pad_value = -2
    n_input_bins = n_bins

if ADV and DAB:
    raise ValueError("ADV and DAB cannot be both True.")
DAB_separate_optim = True if DAB > 1 else False

In [ ]:
dataset_name = config.dataset_name
save_dir = Path(f"./save/dev_{dataset_name}-{time.strftime('%b%d-%H-%M')}/")
save_dir.mkdir(parents=True, exist_ok=True)
print(f"save to {save_dir}")
logger = scg.logger
scg.utils.add_file_handler(logger, save_dir / "run.log")

## Step 2: Load and pre-process data
We follow the standard scGPT data pre-processing pipelines for the cell-type annotation task. Note that since now we have two datasets at hand (i.e., reference and query data), the same pre-prpocessing steps need to be applied to both of them.

In [ ]:
################## MS Dataset ##################

if dataset_name == "ms":
    data_dir = Path("") # path to ms dataset
    adata = sc.read(data_dir / "c_data.h5ad")
    adata_test = sc.read(data_dir / "filtered_ms_adata.h5ad")
    adata.obs["celltype"] = adata.obs["Factor Value[inferred cell type - authors labels]"].astype("category")
    adata_test.obs["celltype"] = adata_test.obs["Factor Value[inferred cell type - authors labels]"].astype("category")
    adata.obs["batch_id"]  = adata.obs["str_batch"] = "0"
    adata_test.obs["batch_id"]  = adata_test.obs["str_batch"] = "1"          
    adata.var.set_index(adata.var["gene_name"], inplace=True)
    adata_test.var.set_index(adata.var["gene_name"], inplace=True)
    data_is_raw = False
    filter_gene_by_counts = False
    adata_test_raw = adata_test.copy()
    adata = adata.concatenate(adata_test, batch_key="str_batch")

################## Pancreas Dataset ##################
if dataset_name == "pancreas":
    data_dir = Path("") # path to pancreas dataset
    adata = sc.read(data_dir / "demo_train.h5ad")
    adata_test = sc.read(data_dir / "demo_test.h5ad")

    adata.obs["celltype"] = adata.obs["Celltype"].astype("category")
    adata_test.obs["celltype"] = adata_test.obs["Celltype"].astype("category")

    adata.obs["batch_id"] = "0"
    adata_test.obs["batch_id"] = "1"

    adata.obs["str_batch"] = adata.obs["batch_id"]
    adata_test.obs["str_batch"] = adata_test.obs["batch_id"]

    adata.var.set_index(adata.var["Gene Symbol"], inplace=True)
    adata_test.var.set_index(adata.var["Gene Symbol"], inplace=True)

    adata.var["gene_name"] = adata.var["Gene Symbol"]
    adata_test.var["gene_name"] = adata_test.var["Gene Symbol"]

    data_is_raw = False
    filter_gene_by_counts = False
    adata_test_raw = adata_test.copy()
    adata = adata.concatenate(adata_test, batch_key="batch_id") 
                   
# make the batch category column
batch_id_labels = adata.obs["str_batch"].astype("category").cat.codes.values
adata.obs["batch_id"] = batch_id_labels
celltype_id_labels = adata.obs["celltype"].astype("category").cat.codes.values
celltypes = adata.obs["celltype"].unique()
num_types = len(np.unique(celltype_id_labels))
id2type = dict(enumerate(adata.obs["celltype"].astype("category").cat.categories))
adata.obs["celltype_id"] = celltype_id_labels
adata.var["gene_name"] = adata.var.index.tolist()

In [ ]:
print(f'Cell Type IDs: {id2type}')

In [ ]:
if config.load_model is not None:
    model_dir = Path(config.load_model)
    model_config_file = model_dir / "args.json"
    model_file = model_dir / "best_model.pt"
    vocab_file = model_dir / "vocab.json"

    vocab = GeneVocab.from_file(vocab_file)
    shutil.copy(vocab_file, save_dir / "vocab.json")
    for s in special_tokens:
        if s not in vocab:
            vocab.append_token(s)

    adata.var["id_in_vocab"] = [
        1 if gene in vocab else -1 for gene in adata.var["gene_name"]
    ]
    gene_ids_in_vocab = np.array(adata.var["id_in_vocab"])
    logger.info(
        f"match {np.sum(gene_ids_in_vocab >= 0)}/{len(gene_ids_in_vocab)} genes "
        f"in vocabulary of size {len(vocab)}."
    )
    adata = adata[:, adata.var["id_in_vocab"] >= 0]

    # model
    with open(model_config_file, "r") as f:
        model_configs = json.load(f)
    logger.info(
        f"Resume model from {model_file}, the model args will override the "
        f"config {model_config_file}."
    )
    embsize = model_configs["embsize"]
    nhead = model_configs["nheads"]
    d_hid = model_configs["d_hid"]
    nlayers = model_configs["nlayers"]
    n_layers_cls = model_configs["n_layers_cls"]

In [ ]:
# set up the preprocessor, use the args to config the workflow
preprocessor = Preprocessor(
    use_key="X",  # the key in adata.layers to use as raw data
    filter_gene_by_counts=filter_gene_by_counts,  # step 1
    filter_cell_by_counts=False,  # step 2
    normalize_total=1e4,  # 3. whether to normalize the raw data and to what sum
    result_normed_key="X_normed",  # the key in adata.layers to store the normalized data
    log1p=data_is_raw,  # 4. whether to log1p the normalized data
    result_log1p_key="X_log1p",
    subset_hvg=False,  # 5. whether to subset the raw data to highly variable genes
    hvg_flavor="seurat_v3" if data_is_raw else "cell_ranger",
    binning=n_bins,  # 6. whether to bin the raw data and to what number of bins
    result_binned_key="X_binned",  # the key in adata.layers to store the binned data
)

adata_test = adata[adata.obs["str_batch"] == "1"]
adata = adata[adata.obs["str_batch"] == "0"]

preprocessor(adata, batch_key=None)
preprocessor(adata_test, batch_key=None)

In [ ]:
input_layer_key = {  # the values of this map coorespond to the keys in preprocessing
    "normed_raw": "X_normed",
    "log1p": "X_normed",
    "binned": "X_binned",
}[input_style]
all_counts = (
    adata.layers[input_layer_key].A
    if issparse(adata.layers[input_layer_key])
    else adata.layers[input_layer_key]
)
genes = adata.var["gene_name"].tolist()

celltypes_labels = adata.obs["celltype_id"].tolist()  # make sure count from 0
celltypes_labels = np.array(celltypes_labels)

batch_ids = adata.obs["batch_id"].tolist()
num_batch_types = len(set(batch_ids))
batch_ids = np.array(batch_ids)

(
    train_data,
    valid_data,
    train_celltype_labels,
    valid_celltype_labels,
    train_batch_labels,
    valid_batch_labels,
) = train_test_split(
    all_counts, celltypes_labels, batch_ids, test_size=0.1, shuffle=True
)

In [ ]:
if config.load_model is None:
    vocab = Vocab(
        VocabPybind(genes + special_tokens, None)
    )  # bidirectional lookup [gene <-> int]
vocab.set_default_index(vocab["<pad>"])
gene_ids = np.array(vocab(genes), dtype=int)

In [ ]:
tokenized_train = tokenize_and_pad_batch(
    train_data,
    gene_ids,
    max_len=max_seq_len,
    vocab=vocab,
    pad_token=pad_token,
    pad_value=pad_value,
    append_cls=True,  # append <cls> token at the beginning
    include_zero_gene=include_zero_gene,
)
tokenized_valid = tokenize_and_pad_batch(
    valid_data,
    gene_ids,
    max_len=max_seq_len,
    vocab=vocab,
    pad_token=pad_token,
    pad_value=pad_value,
    append_cls=True,
    include_zero_gene=include_zero_gene,
)
logger.info(
    f"train set number of samples: {tokenized_train['genes'].shape[0]}, "
    f"\n\t feature length: {tokenized_train['genes'].shape[1]}"
)
logger.info(
    f"valid set number of samples: {tokenized_valid['genes'].shape[0]}, "
    f"\n\t feature length: {tokenized_valid['genes'].shape[1]}"
)

In [ ]:
def prepare_data(sort_seq_batch=False) -> Tuple[Dict[str, torch.Tensor]]:
    masked_values_train = random_mask_value(
        tokenized_train["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )
    masked_values_valid = random_mask_value(
        tokenized_valid["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )
    print(
        f"random masking at epoch {epoch:3d}, ratio of masked values in train: ",
        f"{(masked_values_train == mask_value).sum() / (masked_values_train - pad_value).count_nonzero():.4f}",
    )

    input_gene_ids_train, input_gene_ids_valid = (
        tokenized_train["genes"],
        tokenized_valid["genes"],
    )
    input_values_train, input_values_valid = masked_values_train, masked_values_valid
    target_values_train, target_values_valid = (
        tokenized_train["values"],
        tokenized_valid["values"],
    )

    tensor_batch_labels_train = torch.from_numpy(train_batch_labels).long()
    tensor_batch_labels_valid = torch.from_numpy(valid_batch_labels).long()

    tensor_celltype_labels_train = torch.from_numpy(train_celltype_labels).long()
    tensor_celltype_labels_valid = torch.from_numpy(valid_celltype_labels).long()

    if sort_seq_batch:
        train_sort_ids = np.argsort(train_batch_labels)
        input_gene_ids_train = input_gene_ids_train[train_sort_ids]
        input_values_train = input_values_train[train_sort_ids]
        target_values_train = target_values_train[train_sort_ids]
        tensor_batch_labels_train = tensor_batch_labels_train[train_sort_ids]
        tensor_celltype_labels_train = tensor_celltype_labels_train[train_sort_ids]

        valid_sort_ids = np.argsort(valid_batch_labels)
        input_gene_ids_valid = input_gene_ids_valid[valid_sort_ids]
        input_values_valid = input_values_valid[valid_sort_ids]
        target_values_valid = target_values_valid[valid_sort_ids]
        tensor_batch_labels_valid = tensor_batch_labels_valid[valid_sort_ids]
        tensor_celltype_labels_valid = tensor_celltype_labels_valid[valid_sort_ids]

    train_data_pt = {
        "gene_ids": input_gene_ids_train,
        "values": input_values_train,
        "target_values": target_values_train,
        "batch_labels": tensor_batch_labels_train,
        "celltype_labels": tensor_celltype_labels_train,
    }
    valid_data_pt = {
        "gene_ids": input_gene_ids_valid,
        "values": input_values_valid,
        "target_values": target_values_valid,
        "batch_labels": tensor_batch_labels_valid,
        "celltype_labels": tensor_celltype_labels_valid,
    }

    return train_data_pt, valid_data_pt


# dataset
class SeqDataset(Dataset):
    def __init__(self, data: Dict[str, torch.Tensor]):
        self.data = data

    def __len__(self):
        return self.data["gene_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.data.items()}

In [ ]:
# data_loader
def prepare_dataloader(
    data_pt: Dict[str, torch.Tensor],
    batch_size: int,
    shuffle: bool = False,
    intra_domain_shuffle: bool = False,
    drop_last: bool = False,
    num_workers: int = 0,
) -> DataLoader:
    if num_workers == 0:
        num_workers = min(len(os.sched_getaffinity(0)), batch_size // 2)

    dataset = SeqDataset(data_pt)

    if per_seq_batch_sample:
        # find the indices of samples in each seq batch
        subsets = []
        batch_labels_array = data_pt["batch_labels"].numpy()
        for batch_label in np.unique(batch_labels_array):
            batch_indices = np.where(batch_labels_array == batch_label)[0].tolist()
            subsets.append(batch_indices)
        data_loader = DataLoader(
            dataset=dataset,
            batch_sampler=SubsetsBatchSampler(
                subsets,
                batch_size,
                intra_subset_shuffle=intra_domain_shuffle,
                inter_subset_shuffle=shuffle,
                drop_last=drop_last,
            ),
            num_workers=num_workers,
            pin_memory=True,
        )
        return data_loader

    data_loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,
    )
    return data_loader

## Step 3: Load the pre-trained scGPT model

In [ ]:
def initialize_weights_normal(model: nn.Module, mean: float = 0.0, std: float = 0.02):
    """
    Initialize all model weights using a normal distribution N(mean, std).
    
    Args:
        model: PyTorch model
        mean: Mean of the normal distribution (default: 0.0)
        std: Standard deviation of the normal distribution (default: 0.02)
    """
    for module in model.modules():
        # Linear layers
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=mean, std=std)
            if module.bias is not None:
                module.bias.data.zero_()
        
        # Embedding layers
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=mean, std=std)
            if hasattr(module, 'padding_idx') and module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        
        # LayerNorm
        elif isinstance(module, nn.LayerNorm):
            module.weight.data.fill_(1.0)  # gamma
            module.bias.data.zero_()       # beta
        
        # Conv layers
        elif isinstance(module, (nn.Conv1d, nn.Conv2d, nn.Conv3d)):
            module.weight.data.normal_(mean=mean, std=std)
            if module.bias is not None:
                module.bias.data.zero_()
        
        # Any other layer with parameters
        else:
            for param_name, param in module.named_parameters(recurse=False):
                if 'weight' in param_name:
                    # Apply normal initialization to any other weight parameters
                    param.data.normal_(mean=mean, std=std)
                elif 'bias' in param_name:
                    # Zero initialization for any bias parameters
                    param.data.zero_()


def random_init_scgpt(model, mean=0.0, std=0.02):
    """
    Random initialization for scGPT model.
    
    Args:
        model: scGPT model
        mean: Mean of normal distribution
        std: Standard deviation of normal distribution
    
    Returns:
        The randomly initialized model
    """
    print("Applying standardized random initialization to scGPT model...")
    initialize_weights_normal(model, mean=mean, std=std)
    return model


def verify_standardized_initialization(model: nn.Module, 
                                      normal_mean: float = 0.0, 
                                      normal_std: float = 0.02,
                                      tolerance: float = 0.01) -> Dict[str, Any]:
    """
    Verify that model weights have been initialized according to our standardized scheme:
    - Linear layers: weights ~ N(mean, std), biases = 0
    - Embedding layers: weights ~ N(mean, std)
    - LayerNorm: gamma = 1.0, beta = 0.0
    - All other weight parameters: ~ N(mean, std)
    
    Args:
        model: PyTorch model
        normal_mean: Expected mean for normal distribution (default: 0.0)
        normal_std: Expected std for normal distribution (default: 0.02)
        tolerance: Tolerance for mean/std checks
        
    Returns:
        Dictionary with verification results
    """
    results = {
        "all_layers_verified": True,
        "layers_checked": 0,
        "issues": [],
        "stats": {}
    }
    
    # Counters for different layer types
    layer_counts = {
        "linear": 0,
        "embedding": 0,
        "layernorm": 0,
        "other": 0
    }
    
    # Collect weights by layer type for statistical analysis
    layer_weights = {
        "linear": [],
        "embedding": [],
        "other": []
    }
    
    # List to track verification of each layer
    layer_verifications = []
    
    print(f"\nVerifying initialization against standardized scheme:")
    print(f"- Linear layers: weights ~ N({normal_mean}, {normal_std}), biases = 0")
    print(f"- Embedding layers: weights ~ N({normal_mean}, {normal_std})")
    print(f"- LayerNorm: gamma = 1.0, beta = 0.0")
    print(f"- Other weight parameters: ~ N({normal_mean}, {normal_std})")
    print("-" * 80)
    
    # Check each named module
    for name, module in model.named_modules():
        # Skip top-level container modules to avoid redundant parameter checking
        if len(name.split('.')) <= 1 and not isinstance(module, (nn.Linear, nn.Embedding, nn.LayerNorm)):
            continue
            
        if isinstance(module, nn.Linear):
            # Check linear layer weights
            layer_counts["linear"] += 1
            weights = module.weight.detach().cpu().numpy().flatten()
            layer_weights["linear"].extend(weights)
            
            # Calculate statistics
            w_mean = np.mean(weights)
            w_std = np.std(weights)
            
            # Check if weights match expected normal distribution
            weights_correct = (abs(w_mean - normal_mean) < tolerance and 
                              abs(w_std - normal_std) < tolerance * 3)
            
            # Check if biases are zeros
            bias_correct = True
            if module.bias is not None:
                biases = module.bias.detach().cpu().numpy().flatten()
                bias_mean = np.mean(np.abs(biases))
                bias_correct = bias_mean < tolerance / 10
            
            layer_correct = weights_correct and bias_correct
            
            # Log verification result
            layer_verifications.append({
                "name": name,
                "type": "linear",
                "weights_correct": weights_correct,
                "bias_correct": bias_correct,
                "w_mean": w_mean,
                "w_std": w_std,
                "correct": layer_correct
            })
            
            if not layer_correct:
                results["issues"].append({
                    "layer": name,
                    "type": "linear",
                    "weights_mean": w_mean,
                    "weights_std": w_std,
                    "expected_mean": normal_mean,
                    "expected_std": normal_std,
                    "bias_correct": bias_correct
                })
                results["all_layers_verified"] = False
                
        elif isinstance(module, nn.Embedding):
            # Check embedding layer weights
            layer_counts["embedding"] += 1
            weights = module.weight.detach().cpu().numpy()
            
            # Exclude padding token if it exists
            if hasattr(module, 'padding_idx') and module.padding_idx is not None:
                if module.padding_idx < weights.shape[0]:
                    # Create a mask to exclude padding token
                    mask = np.ones(weights.shape[0], dtype=bool)
                    mask[module.padding_idx] = False
                    weights = weights[mask].flatten()
                else:
                    weights = weights.flatten()
            else:
                weights = weights.flatten()
                
            layer_weights["embedding"].extend(weights)
            
            # Calculate statistics
            w_mean = np.mean(weights)
            w_std = np.std(weights)
            
            # Check if weights match expected normal distribution
            weights_correct = (abs(w_mean - normal_mean) < tolerance and 
                              abs(w_std - normal_std) < tolerance * 3)
            
            layer_verifications.append({
                "name": name,
                "type": "embedding",
                "weights_correct": weights_correct,
                "w_mean": w_mean,
                "w_std": w_std,
                "correct": weights_correct
            })
            
            if not weights_correct:
                results["issues"].append({
                    "layer": name,
                    "type": "embedding",
                    "weights_mean": w_mean,
                    "weights_std": w_std,
                    "expected_mean": normal_mean,
                    "expected_std": normal_std
                })
                results["all_layers_verified"] = False
                
        elif isinstance(module, nn.LayerNorm):
            # Check LayerNorm parameters
            layer_counts["layernorm"] += 1
            
            # LayerNorm should have gamma=1 and beta=0
            gamma = module.weight.detach().cpu().numpy().flatten()
            beta = module.bias.detach().cpu().numpy().flatten()
            
            gamma_correct = np.allclose(gamma, 1.0, atol=tolerance)
            beta_correct = np.allclose(beta, 0.0, atol=tolerance)
            
            layer_correct = gamma_correct and beta_correct
            
            layer_verifications.append({
                "name": name,
                "type": "layernorm",
                "gamma_correct": gamma_correct,
                "beta_correct": beta_correct,
                "gamma_mean": np.mean(gamma),
                "beta_mean": np.mean(beta),
                "correct": layer_correct
            })
            
            if not layer_correct:
                results["issues"].append({
                    "layer": name,
                    "type": "layernorm",
                    "gamma_mean": np.mean(gamma),
                    "gamma_expected": 1.0,
                    "beta_mean": np.mean(beta),
                    "beta_expected": 0.0
                })
                results["all_layers_verified"] = False
                
        elif len(list(module.parameters(recurse=False))) > 0:
            # Check parameters of other layer types
            for param_name, param in module.named_parameters(recurse=False):
                # Only check weight parameters, not biases
                if 'weight' in param_name:
                    layer_counts["other"] += 1
                    weights = param.detach().cpu().numpy().flatten()
                    layer_weights["other"].extend(weights)
                    
                    w_mean = np.mean(weights)
                    w_std = np.std(weights)
                    
                    # Check if weights match expected normal distribution
                    weights_correct = (abs(w_mean - normal_mean) < tolerance and 
                                      abs(w_std - normal_std) < tolerance * 3)
                    
                    full_name = f"{name}.{param_name}"
                    layer_verifications.append({
                        "name": full_name,
                        "type": "other",
                        "weights_correct": weights_correct,
                        "w_mean": w_mean,
                        "w_std": w_std,
                        "correct": weights_correct
                    })
                    
                    if not weights_correct:
                        results["issues"].append({
                            "layer": full_name,
                            "type": "other",
                            "weights_mean": w_mean,
                            "weights_std": w_std,
                            "expected_mean": normal_mean,
                            "expected_std": normal_std
                        })
                        results["all_layers_verified"] = False
                
                # Check biases
                elif 'bias' in param_name:
                    biases = param.detach().cpu().numpy().flatten()
                    bias_mean = np.mean(np.abs(biases))
                    bias_correct = bias_mean < tolerance / 10
                    
                    full_name = f"{name}.{param_name}"
                    if not bias_correct:
                        results["issues"].append({
                            "layer": full_name,
                            "type": "bias",
                            "bias_mean": bias_mean,
                            "expected": 0.0
                        })
                        results["all_layers_verified"] = False
        
        results["layers_checked"] += 1
    
    # Calculate overall statistics for each layer type
    for layer_type, weights in layer_weights.items():
        if weights:
            results["stats"][layer_type] = {
                "mean": np.mean(weights),
                "std": np.std(weights),
                "min": np.min(weights),
                "max": np.max(weights),
                "count": len(weights)
            }
    
    # Count how many layers passed verification
    correct_layers = sum(1 for v in layer_verifications if v["correct"])
    total_layers = len(layer_verifications)
    
    # Print verification results
    print(f"\nVerification Summary:")
    print(f"- Linear layers: {layer_counts['linear']}")
    print(f"- Embedding layers: {layer_counts['embedding']}")
    print(f"- LayerNorm layers: {layer_counts['layernorm']}")
    print(f"- Other parameter layers: {layer_counts['other']}")
    print(f"- Total layers checked: {results['layers_checked']}")
    print(f"- Layers correctly initialized: {correct_layers}/{total_layers}")
    
    print("\nWeight Statistics:")
    for layer_type, stats in results["stats"].items():
        print(f"- {layer_type.capitalize()} layers: mean={stats['mean']:.6f}, std={stats['std']:.6f}")
    
    # Plot weight distribution
    plt.figure(figsize=(12, 8))
    
    # Plot histograms for each layer type
    plt.subplot(2, 2, 1)
    for layer_type, weights in layer_weights.items():
        if weights:
            plt.hist(weights, bins=50, alpha=0.5, label=f'{layer_type.capitalize()}')
    plt.title('Weight Distribution by Layer Type')
    plt.xlabel('Weight Value')
    plt.ylabel('Frequency')
    plt.legend()
    
    # Plot Q-Q plot for linear layer weights
    if layer_weights["linear"]:
        plt.subplot(2, 2, 2)
        from scipy import stats
        linear_sample = np.random.choice(layer_weights["linear"], 
                                         size=min(1000, len(layer_weights["linear"])))
        stats.probplot(linear_sample, dist="norm", plot=plt)
        plt.title('Q-Q Plot (Linear Layer Weights)')
    
    # Plot initialization correctness by layer type
    plt.subplot(2, 2, 3)
    layer_types = ["linear", "embedding", "layernorm", "other"]
    correct_by_type = {layer_type: 0 for layer_type in layer_types}
    total_by_type = {layer_type: 0 for layer_type in layer_types}
    
    for v in layer_verifications:
        if v["type"] in total_by_type:
            total_by_type[v["type"]] += 1
            if v["correct"]:
                correct_by_type[v["type"]] += 1
    
    # Calculate percentage correct for each type
    pct_correct = []
    for layer_type in layer_types:
        if total_by_type[layer_type] > 0:
            pct = 100 * correct_by_type[layer_type] / total_by_type[layer_type]
            pct_correct.append(pct)
        else:
            pct_correct.append(0)
    
    plt.bar(layer_types, pct_correct)
    plt.title('Initialization Correctness by Layer Type')
    plt.xlabel('Layer Type')
    plt.ylabel('Percent Correct')
    plt.ylim(0, 105)
    
    # Add text labels
    for i, pct in enumerate(pct_correct):
        if total_by_type[layer_types[i]] > 0:
            plt.text(i, pct + 2, f"{pct:.1f}%", ha='center')
            plt.text(i, pct/2, f"{correct_by_type[layer_types[i]]}/{total_by_type[layer_types[i]]}", 
                    ha='center', color='white')
    
    # Overall correctness gauge
    plt.subplot(2, 2, 4)
    overall_pct = 100 * correct_layers / total_layers if total_layers > 0 else 0
    plt.pie([overall_pct, 100-overall_pct], 
            labels=[f'Correct ({correct_layers})', f'Issues ({total_layers-correct_layers})'],
            colors=['green', 'red'], autopct='%1.1f%%', 
            startangle=90)
    plt.axis('equal')
    plt.title('Overall Initialization Correctness')
    
    plt.tight_layout()
    plt.savefig('initialization_verification.png')
    print(f"\nVisualization saved to: initialization_verification.png")
    
    # Final verdict
    if results["all_layers_verified"]:
        print("\n✅ All layers are correctly initialized according to the standardized scheme!")
    else:
        print(f"\n⚠️ Found {len(results['issues'])} issues with initialization.")
        print("Here are the top issues:")
        for i, issue in enumerate(results["issues"][:5]):  # Show top 5 issues
            print(f"  {i+1}. {issue['layer']} ({issue['type']}): ", end="")
            if issue['type'] == 'layernorm':
                print(f"gamma={issue['gamma_mean']:.4f} (expected: 1.0), "
                      f"beta={issue['beta_mean']:.4f} (expected: 0.0)")
            elif issue['type'] == 'bias':
                print(f"bias_mean={issue['bias_mean']:.6f} (expected: 0.0)")
            else:
                print(f"mean={issue['weights_mean']:.6f} (expected: {normal_mean}), "
                      f"std={issue['weights_std']:.6f} (expected: {normal_std})")
        if len(results["issues"]) > 5:
            print(f"  ... and {len(results['issues'])-5} more issues.")
    
    return results

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ntokens = len(vocab)  # size of vocabulary
model = TransformerModel(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    nlayers_cls=3,
    n_cls=num_types if CLS else 1,
    vocab=vocab,
    dropout=dropout,
    output_attentions=True,
    pad_token=pad_token,
    pad_value=pad_value,
    do_mvc=MVC,
    do_dab=DAB,
    use_batch_labels=INPUT_BATCH_LABELS,
    num_batch_labels=num_batch_types,
    domain_spec_batchnorm=config.DSBN,
    input_emb_style=input_emb_style,
    n_input_bins=n_input_bins,
    cell_emb_style=cell_emb_style,
    mvc_decoder_style=mvc_decoder_style,
    ecs_threshold=ecs_threshold,
    explicit_zero_prob=explicit_zero_prob,
    use_fast_transformer=fast_transformer,
    fast_transformer_backend=fast_transformer_backend,
    pre_norm=config.pre_norm,
)
if config.load_model is not None:
    try:
        model.load_state_dict(torch.load(model_file))
        logger.info(f"Loading all model params from {model_file}")
    except:
        # only load params that are in the model and match the size
        model_dict = model.state_dict()
        pretrained_dict = torch.load(model_file)
        pretrained_dict = {
            k: v
            for k, v in pretrained_dict.items()
            if k in model_dict and v.shape == model_dict[k].shape
        }
        for k, v in pretrained_dict.items():
            logger.info(f"Loading params {k} with shape {v.shape}")
        model_dict.update(pretrained_dict)
        model.load_state_dict(model_dict)

if task_name == 'random_init':
    model = random_init_scgpt(model)
    # Verify initialization
    results = verify_standardized_initialization(model)
    
    # Print results summary
    print(f"Initialization verification complete:")
    print(f"- Layers checked: {results['layers_checked']}")
    print(f"- All layers verified: {results['all_layers_verified']}")
    
    if not results['all_layers_verified']:
        print(f"- Found {len(results['anomalies'])} anomalies")
        for i, anomaly in enumerate(results['anomalies'][:5]):  # Show only first 5
            print(f"  Anomaly {i+1}: {anomaly}")
        
        if len(results['anomalies']) > 5:
            print(f"  ... and {len(results['anomalies'])-5} more anomalies.")

pre_freeze_param_count = sum(dict((p.data_ptr(), p.numel()) for p in model.parameters() if p.requires_grad).values())

# Freeze all pre-decoder weights
for name, para in model.named_parameters():
    print("-"*20)
    print(f"name: {name}")
    if config.freeze and "encoder" in name and "transformer_encoder" not in name:
    # if config.freeze and "encoder" in name:
        print(f"freezing weights for: {name}")
        para.requires_grad = False

post_freeze_param_count = sum(dict((p.data_ptr(), p.numel()) for p in model.parameters() if p.requires_grad).values())

logger.info(f"Total Pre freeze Params {(pre_freeze_param_count )}")
logger.info(f"Total Post freeze Params {(post_freeze_param_count )}")
wandb.log(
        {
            "info/pre_freeze_param_count": pre_freeze_param_count,
            "info/post_freeze_param_count": post_freeze_param_count,
        },
)

model.to(device)
wandb.watch(model)

if ADV:
    discriminator = AdversarialDiscriminator(
        d_model=embsize,
        n_cls=num_batch_types,
    ).to(device)


In [ ]:
criterion = masked_mse_loss
criterion_cls = nn.CrossEntropyLoss()
criterion_dab = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=lr, eps=1e-4 if config.amp else 1e-8
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, schedule_interval, gamma=config.schedule_ratio
)
if DAB_separate_optim:
    optimizer_dab = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler_dab = torch.optim.lr_scheduler.StepLR(
        optimizer_dab, schedule_interval, gamma=config.schedule_ratio
    )
if ADV:
    criterion_adv = nn.CrossEntropyLoss()  # consider using label smoothing
    optimizer_E = torch.optim.Adam(model.parameters(), lr=lr_ADV)
    scheduler_E = torch.optim.lr_scheduler.StepLR(
        optimizer_E, schedule_interval, gamma=config.schedule_ratio
    )
    optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr_ADV)
    scheduler_D = torch.optim.lr_scheduler.StepLR(
        optimizer_D, schedule_interval, gamma=config.schedule_ratio
    )

scaler = torch.cuda.amp.GradScaler(enabled=config.amp)

In [ ]:
def train(model: nn.Module, loader: DataLoader) -> None:
    """
    Train the model for one epoch.
    """
    model.train()
    (
        total_loss,
        total_mse,
        total_cls,
        total_cce,
        total_mvc,
        total_ecs,
        total_dab,
        total_adv_E,
        total_adv_D,
        total_zero_log_prob,
        total_mvc_zero_log_prob,
    ) = (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
    total_error = 0.0
    start_time = time.time()

    num_batches = len(loader)
    for batch, batch_data in enumerate(loader):
        input_gene_ids = batch_data["gene_ids"].to(device)
        input_values = batch_data["values"].to(device)
        target_values = batch_data["target_values"].to(device)
        batch_labels = batch_data["batch_labels"].to(device)
        celltype_labels = batch_data["celltype_labels"].to(device)

        src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])
        with torch.cuda.amp.autocast(enabled=config.amp):
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
                CLS=CLS,
                CCE=CCE,
                MVC=MVC,
                ECS=ECS,
                do_sample=do_sample_in_train,
                #generative_training=False
            )

            masked_positions = input_values.eq(mask_value)  # the postions to predict
            loss = 0.0
            metrics_to_log = {}
            if MLM:
                loss_mse = criterion(
                    output_dict["mlm_output"], target_values, masked_positions
                )
                loss = loss + loss_mse
                metrics_to_log = {"train/mse": loss_mse.item()}
            if explicit_zero_prob:
                loss_zero_log_prob = criterion_neg_log_bernoulli(
                    output_dict["mlm_zero_probs"], target_values, masked_positions
                )
                loss = loss + loss_zero_log_prob
                metrics_to_log.update({"train/nzlp": loss_zero_log_prob.item()})
            if CLS:
                loss_cls = criterion_cls(output_dict["cls_output"], celltype_labels)
                loss = loss + loss_cls
                metrics_to_log.update({"train/cls": loss_cls.item()})

                error_rate = 1 - (
                    (output_dict["cls_output"].argmax(1) == celltype_labels)
                    .sum()
                    .item()
                ) / celltype_labels.size(0)
            if CCE:
                loss_cce = 10 * output_dict["loss_cce"]
                loss = loss + loss_cce
                metrics_to_log.update({"train/cce": loss_cce.item()})
            if MVC:
                loss_mvc = criterion(
                    output_dict["mvc_output"], target_values, masked_positions
                )
                loss = loss + loss_mvc
                metrics_to_log.update({"train/mvc": loss_mvc.item()})
            if MVC and explicit_zero_prob:
                loss_mvc_zero_log_prob = criterion_neg_log_bernoulli(
                    output_dict["mvc_zero_probs"], target_values, masked_positions
                )
                loss = loss + loss_mvc_zero_log_prob
                metrics_to_log.update({"train/mvc_nzlp": loss_mvc_zero_log_prob.item()})
            if ECS:
                loss_ecs = 10 * output_dict["loss_ecs"]
                loss = loss + loss_ecs
                metrics_to_log.update({"train/ecs": loss_ecs.item()})
            if DAB:
                # try weighting and separate optimizer
                loss_dab = criterion_dab(output_dict["dab_output"], batch_labels)
                loss = loss + dab_weight * loss_dab
                metrics_to_log.update({"train/dab": loss_dab.item()})

        model.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        with warnings.catch_warnings(record=True) as w:
            warnings.filterwarnings("always")
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
                error_if_nonfinite=False if scaler.is_enabled() else True,
            )
            if len(w) > 0:
                logger.warning(
                    f"Found infinite gradient. This may be caused by the gradient "
                    f"scaler. The current scale is {scaler.get_scale()}. This warning "
                    "can be ignored if no longer occurs after autoscaling of the scaler."
                )
        scaler.step(optimizer)
        scaler.update()

        if ADV:
            # rerun the model for adversarial training
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
                CLS=CLS,
                CCE=CCE,
                MVC=MVC,
                ECS=ECS,
                do_sample=do_sample_in_train,
                #generative_training=False
            )

            # TRAINING DISCRIMINATOR
            loss_adv_D = criterion_adv(
                discriminator(output_dict["cell_emb"].detach()), batch_labels
            )
            if epoch > adv_D_delay_epochs:
                discriminator.zero_grad()
                loss_adv_D.backward()
                optimizer_D.step()

            # TRAINING ENCODER
            loss_adv_E = -criterion_adv(
                discriminator(output_dict["cell_emb"]), batch_labels
            )
            # NOTE: the loss is negative here because we want to maximize
            # the cross_entropy_loss, in other words, disguise against the discriminator
            if epoch > adv_E_delay_epochs:
                model.zero_grad()
                discriminator.zero_grad()
                loss_adv_E.backward()
                optimizer_E.step()

        wandb.log(metrics_to_log)

        total_loss += loss.item()
        total_mse += loss_mse.item() if MLM else 0.0
        total_cls += loss_cls.item() if CLS else 0.0
        total_cce += loss_cce.item() if CCE else 0.0
        total_mvc += loss_mvc.item() if MVC else 0.0
        total_ecs += loss_ecs.item() if ECS else 0.0
        total_dab += loss_dab.item() if DAB else 0.0
        total_adv_E += loss_adv_E.item() if ADV else 0.0
        total_adv_D += loss_adv_D.item() if ADV else 0.0
        total_zero_log_prob += loss_zero_log_prob.item() if explicit_zero_prob else 0.0
        total_mvc_zero_log_prob += (
            loss_mvc_zero_log_prob.item() if MVC and explicit_zero_prob else 0.0
        )
        total_error += error_rate
        if batch % log_interval == 0 and batch > 0:
            lr = scheduler.get_last_lr()[0]
            ms_per_batch = (time.time() - start_time) * 1000 / log_interval
            cur_loss = total_loss / log_interval
            cur_mse = total_mse / log_interval
            cur_cls = total_cls / log_interval if CLS else 0.0
            cur_cce = total_cce / log_interval if CCE else 0.0
            cur_mvc = total_mvc / log_interval if MVC else 0.0
            cur_ecs = total_ecs / log_interval if ECS else 0.0
            cur_dab = total_dab / log_interval if DAB else 0.0
            cur_adv_E = total_adv_E / log_interval if ADV else 0.0
            cur_adv_D = total_adv_D / log_interval if ADV else 0.0
            cur_zero_log_prob = (
                total_zero_log_prob / log_interval if explicit_zero_prob else 0.0
            )
            cur_mvc_zero_log_prob = (
                total_mvc_zero_log_prob / log_interval
                if MVC and explicit_zero_prob
                else 0.0
            )
            cur_error = total_error / log_interval
            # ppl = math.exp(cur_loss)
            logger.info(
                f"| epoch {epoch:3d} | {batch:3d}/{num_batches:3d} batches | "
                f"lr {lr:05.4f} | ms/batch {ms_per_batch:5.2f} | "
                f"loss {cur_loss:5.2f} | "
                + (f"mse {cur_mse:5.2f} | mre {cur_error:5.2f} |" if MLM else "")
                + (f"cls {cur_cls:5.2f} | " if CLS else "")
                + (f"err {cur_error:5.2f} | " if CLS else "")
                + (f"cce {cur_cce:5.2f} |" if CCE else "")
                + (f"mvc {cur_mvc:5.2f} |" if MVC else "")
                + (f"ecs {cur_ecs:5.2f} |" if ECS else "")
                + (f"dab {cur_dab:5.2f} |" if DAB else "")
                + (f"adv_E {cur_adv_E:5.2f} |" if ADV else "")
                + (f"adv_D {cur_adv_D:5.2f} |" if ADV else "")
                + (f"nzlp {cur_zero_log_prob:5.2f} |" if explicit_zero_prob else "")
                + (
                    f"mvc_nzlp {cur_mvc_zero_log_prob:5.2f} |"
                    if MVC and explicit_zero_prob
                    else ""
                )
            )
            total_loss = 0
            total_mse = 0
            total_cls = 0
            total_cce = 0
            total_mvc = 0
            total_ecs = 0
            total_dab = 0
            total_adv_E = 0
            total_adv_D = 0
            total_zero_log_prob = 0
            total_mvc_zero_log_prob = 0
            total_error = 0
            start_time = time.time()


def define_wandb_metrcis():
    wandb.define_metric("valid/mse", summary="min", step_metric="epoch")
    wandb.define_metric("valid/mre", summary="min", step_metric="epoch")
    wandb.define_metric("valid/dab", summary="min", step_metric="epoch")
    wandb.define_metric("valid/sum_mse_dab", summary="min", step_metric="epoch")
    wandb.define_metric("test/avg_bio", summary="max")


def evaluate(model: nn.Module, loader: DataLoader, return_raw: bool = False) -> float:
    """
    Evaluate the model on the evaluation data.
    """
    model.eval()
    total_loss = 0.0
    total_error = 0.0
    total_dab = 0.0
    total_num = 0
    predictions = []
    with torch.no_grad():
        for batch_data in loader:
            input_gene_ids = batch_data["gene_ids"].to(device)
            input_values = batch_data["values"].to(device)
            target_values = batch_data["target_values"].to(device)
            batch_labels = batch_data["batch_labels"].to(device)
            celltype_labels = batch_data["celltype_labels"].to(device)

            src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])
            with torch.cuda.amp.autocast(enabled=config.amp):
                output_dict = model(
                    input_gene_ids,
                    input_values,
                    src_key_padding_mask=src_key_padding_mask,
                    batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
                    CLS=CLS,  # evaluation does not need CLS or CCE
                    CCE=False,
                    MVC=False,
                    ECS=False,
                    do_sample=do_sample_in_train,
                    #generative_training = False,
                )
                output_values = output_dict["cls_output"]
                loss = criterion_cls(output_values, celltype_labels)

                if DAB:
                    loss_dab = criterion_dab(output_dict["dab_output"], batch_labels)

            total_loss += loss.item() * len(input_gene_ids)
            accuracy = (output_values.argmax(1) == celltype_labels).sum().item()
            total_error += (1 - accuracy / len(input_gene_ids)) * len(input_gene_ids)
            total_dab += loss_dab.item() * len(input_gene_ids) if DAB else 0.0
            total_num += len(input_gene_ids)
            preds = output_values.argmax(1).cpu().numpy()
            predictions.append(preds)

    wandb.log(
        {
            "valid/mse": total_loss / total_num,
            "valid/err": total_error / total_num,
            "valid/dab": total_dab / total_num,
            "valid/sum_mse_dab": (total_loss + dab_weight * total_dab) / total_num,
            "epoch": epoch,
        },
    )

    if return_raw:
        return np.concatenate(predictions, axis=0)

    return total_loss / total_num, total_error / total_num


## Step 4: Finetune scGPT with task-specific objectives

In [ ]:
best_val_loss = float("inf")
best_avg_bio = 0.0
best_model = None
define_wandb_metrcis()

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()
    train_data_pt, valid_data_pt = prepare_data(sort_seq_batch=per_seq_batch_sample)
    train_loader = prepare_dataloader(
        train_data_pt,
        batch_size=batch_size,
        shuffle=False,
        intra_domain_shuffle=True,
        drop_last=False,
    )
    valid_loader = prepare_dataloader(
        valid_data_pt,
        batch_size=eval_batch_size,
        shuffle=False,
        intra_domain_shuffle=False,
        drop_last=False,
    )

    if config.do_train:
        train(
            model,
            loader=train_loader,
        )
    val_loss, val_err = evaluate(
        model,
        loader=valid_loader,
    )
    elapsed = time.time() - epoch_start_time
    logger.info("-" * 89)
    logger.info(
        f"| end of epoch {epoch:3d} | time: {elapsed:5.2f}s | "
        f"valid loss/mse {val_loss:5.4f} | err {val_err:5.4f}"
    )
    logger.info("-" * 89)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = copy.deepcopy(model)
        best_model_epoch = epoch
        logger.info(f"Best model with score {best_val_loss:5.4f}")

    scheduler.step()
    if DAB_separate_optim:
        scheduler_dab.step()
    if ADV:
        scheduler_D.step()
        scheduler_E.step()

In [ ]:
# %% inference
def test(model: nn.Module, adata: DataLoader) -> float:
    
    all_counts = (
        adata.layers[input_layer_key].A
        if issparse(adata.layers[input_layer_key])
        else adata.layers[input_layer_key]
    )

    celltypes_labels = adata.obs["celltype_id"].tolist()  # make sure count from 0
    celltypes_labels = np.array(celltypes_labels[:499])

    batch_ids = adata.obs["batch_id"].tolist()
    batch_ids = np.array(batch_ids[:499])

    tokenized_test = tokenize_and_pad_batch(
        all_counts[:499],
        gene_ids,
        max_len=max_seq_len,
        vocab=vocab,
        pad_token=pad_token,
        pad_value=pad_value,
        append_cls=True,  # append <cls> token at the beginning
        include_zero_gene=include_zero_gene,
    )

    input_values_test = random_mask_value(
        tokenized_test["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )

    test_data_pt = {
        "gene_ids": tokenized_test["genes"],
        "values": input_values_test,
        "target_values": tokenized_test["values"],
        "batch_labels": torch.from_numpy(batch_ids).long(),
        "celltype_labels": torch.from_numpy(celltypes_labels).long(),
    }
    
    print(len(test_data_pt['gene_ids']))

    test_loader = DataLoader(
        dataset=SeqDataset(test_data_pt),
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=min(len(os.sched_getaffinity(0)), eval_batch_size // 2),
        pin_memory=True,
    )

    model.eval()
    predictions = evaluate(
        model,
        loader=test_loader,
        return_raw=True,
    )

    # compute accuracy, precision, recall, f1
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    accuracy = accuracy_score(celltypes_labels, predictions)
    precision = precision_score(celltypes_labels, predictions, average="macro")
    recall = recall_score(celltypes_labels, predictions, average="macro")
    macro_f1 = f1_score(celltypes_labels, predictions, average="macro")

    logger.info(
        f"Accuracy: {accuracy:.3f}, Precision: {precision:.3f}, Recall: {recall:.3f}, "
        f"Macro F1: {macro_f1:.3f}"
    )

    results = {
        "test/accuracy": accuracy,
        "test/precision": precision,
        "test/recall": recall,
        "test/macro_f1": macro_f1,
    }

    return predictions, celltypes_labels, results

## Step 5: Inference with fine-tuned scGPT model
In the cell-type annotation task, the fine-tuned scGPT predicts cell-type labels for query set as inference. The model performance is evaluated on standard classificaton metrics. Here we visualize the predicted labels over the scGPT cell embeddings, and present the confusion matrix for detailed classification performance on the cell-group level.

In [ ]:
predictions, labels, results = test(best_model, adata_test)
adata_subset = adata_test_raw[:499].copy()  
adata_subset.obs["predictions"] = [id2type[p] for p in predictions]

# plot
palette_ = plt.rcParams["axes.prop_cycle"].by_key()["color"] 
palette_ = plt.rcParams["axes.prop_cycle"].by_key()["color"] + plt.rcParams["axes.prop_cycle"].by_key()["color"] + plt.rcParams["axes.prop_cycle"].by_key()["color"]
palette_ = {c: palette_[i] for i, c in enumerate(celltypes)}

with plt.rc_context({"figure.figsize": (6, 4), "figure.dpi": (300)}):
    sc.pl.umap(
        adata_subset,
        color=["celltype", "predictions"],
        palette=palette_,
        show=False,
    )
    plt.savefig(save_dir / "results.png", dpi=300)

save_dict = {
    "predictions": predictions,
    "labels": labels,
    "results": results,
    "id_maps": id2type
}
with open(save_dir / "results.pkl", "wb") as f:
    pickle.dump(save_dict, f)

results["test/cell_umap"] = wandb.Image(
    str(save_dir / "results.png"),
    caption=f"predictions macro f1 {results['test/macro_f1']:.3f}",
)
wandb.log(results)

In [ ]:
adata_subset.write(save_dir / "adata_test.h5ad")

In [ ]:
from sklearn.metrics import confusion_matrix
celltypes = list(celltypes)
for i in set([id2type[p] for p in predictions]):
    if i not in celltypes:
        celltypes.remove(i)
cm = confusion_matrix(labels, predictions)
cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
cm = pd.DataFrame(cm, index=celltypes[:cm.shape[0]], columns=celltypes[:cm.shape[1]])
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt=".1f", cmap="Blues")
plt.savefig(save_dir / "confusion_matrix.png", dpi=300)

results["test/confusion_matrix"] = wandb.Image(
    str(save_dir / "confusion_matrix.png"),
    caption=f"confusion matrix",
)

In [ ]:
# save the model into the save_dir
torch.save(best_model.state_dict(), save_dir / "best_model.pt")

## Inference Only

In [ ]:
def get_predictions_proba(model, adata, input_layer_key, gene_ids, max_seq_len, vocab, pad_token, pad_value, mask_value, mask_ratio, include_zero_gene, eval_batch_size, device):
    """
    Get class probabilities for each sample in the test set
    """
    model.eval()
    
    # prepare data similar to the test function
    all_counts = (
        adata.layers[input_layer_key].toarray()
        if issparse(adata.layers[input_layer_key])
        else adata.layers[input_layer_key]
    )

    celltypes_labels = adata.obs["celltype_id"].tolist()
    celltypes_labels = np.array(celltypes_labels[:499])

    batch_ids = adata.obs["batch_id"].tolist()
    batch_ids = np.array(batch_ids[:499])

    tokenized_test = tokenize_and_pad_batch(
        all_counts[:499],
        gene_ids,
        max_len=max_seq_len,
        vocab=vocab,
        pad_token=pad_token,
        pad_value=pad_value,
        append_cls=True,
        include_zero_gene=include_zero_gene,
    )

    input_values_test = random_mask_value(
        tokenized_test["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )

    test_data_pt = {
        "gene_ids": tokenized_test["genes"],
        "values": input_values_test,
        "target_values": tokenized_test["values"],
        "batch_labels": torch.from_numpy(batch_ids).long(),
        "celltype_labels": torch.from_numpy(celltypes_labels).long(),
    }

    test_loader = DataLoader(
        dataset=SeqDataset(test_data_pt),
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=min(len(os.sched_getaffinity(0)), eval_batch_size // 2),
        pin_memory=True,
    )

    # get probabilities for each class
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch_data in test_loader:
            input_gene_ids = batch_data["gene_ids"].to(device)
            input_values = batch_data["values"].to(device)
            batch_labels = batch_data["batch_labels"].to(device)
            celltype_labels = batch_data["celltype_labels"].to(device)
            
            src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])
            
            with torch.cuda.amp.autocast(enabled=config.amp):
                output_dict = model(
                    input_gene_ids,
                    input_values,
                    src_key_padding_mask=src_key_padding_mask,
                    batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
                    CLS=True,
                    CCE=False,
                    MVC=False,
                    ECS=False,
                    do_sample=False,
                )
                
                # get probabilities using softmax
                probs = torch.nn.functional.softmax(output_dict["cls_output"], dim=1)
                
                all_probs.append(probs.cpu().numpy())
                all_labels.append(celltype_labels.cpu().numpy())
    
    # concatenate results
    y_probs = np.vstack(all_probs)
    y_true = np.concatenate(all_labels)
    
    return y_probs, y_true, celltypes_labels

def plot_multiclass_roc(y_probs, y_true, class_labels, id2type, save_dir):
    """
    Plot ROC curves for multi-class classification in a one-vs-rest manner.
    
    Args:
        y_probs: Probability predictions (n_samples, n_classes)
        y_true: True labels (n_samples,)
        class_labels: Unique class labels
        id2type: Mapping from class indices to class names
        save_dir: Directory to save plots
    """
    n_classes = len(class_labels)
    
    # binarize the labels for one-vs-rest ROC calculation
    y_true_bin = label_binarize(y_true, classes=class_labels)
    
    # compute ROC curve and ROC area for each class
    fpr = {}
    tpr = {}
    roc_auc = {}
    
    plt.figure(figsize=(12, 10))
    
    # calculate AUC for each class (one-vs-rest)
    for i, class_idx in enumerate(class_labels):
        fpr[i], tpr[i], _ = metrics.roc_curve(y_true_bin[:, i], y_probs[:, class_idx])
        roc_auc[i] = metrics.auc(fpr[i], tpr[i])
        
        # get class name for the legend
        class_name = id2type[class_idx]
        plt.plot(
            fpr[i], 
            tpr[i], 
            lw=2, 
            label=f'{class_name} (AUC = {roc_auc[i]:.2f})'
        )
    
    # plot diagonal
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    
    # set plot parameters
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.title('Multi-class ROC Curves (One-vs-Rest)', fontsize=16)
    plt.legend(loc="lower right", fontsize=10)
    
    # save plot
    plt.tight_layout()
    plt.savefig(save_dir / "multiclass_roc_curves.png", dpi=300)
    
    # calculate and return the macro-average AUC
    macro_roc_auc = metrics.roc_auc_score(y_true_bin, y_probs, multi_class='ovr', average='macro')
    
    return roc_auc, macro_roc_auc

def plot_auc_heatmap(roc_auc, class_labels, id2type, save_dir):
    """
    Plot a heatmap of AUC values for each class
    """
    # create DataFrame for the AUC values
    auc_data = []
    for i, class_idx in enumerate(class_labels):
        auc_data.append({
            'Cell Type': id2type[class_idx], 
            'AUC': roc_auc[i]
        })
    
    auc_df = pd.DataFrame(auc_data).sort_values('AUC', ascending=False)
    
    # plot heatmap
    plt.figure(figsize=(10, len(class_labels) * 0.4 + 2))
    ax = sns.heatmap(
        auc_df.set_index('Cell Type')[['AUC']], 
        annot=True, 
        fmt='.3f', 
        cmap='viridis',
        linewidths=.5,
        cbar_kws={'label': 'AUC Score'}
    )
    plt.title('AUC Scores by Cell Type', fontsize=16)
    plt.tight_layout()
    plt.savefig(save_dir / "auc_heatmap.png", dpi=300)
    
    return auc_df

# func to run the multi-class AUC analysis
def run_multiclass_auc_analysis(model, adata_test, input_layer_key, gene_ids, max_seq_len, 
                               vocab, pad_token, pad_value, mask_value, mask_ratio, 
                               include_zero_gene, eval_batch_size, device, id2type, save_dir):
    """Run the complete multi-class AUC analysis"""
    # get prediction probabilities
    y_probs, y_true, unique_classes = get_predictions_proba(
        model, adata_test, input_layer_key, gene_ids, max_seq_len, 
        vocab, pad_token, pad_value, mask_value, mask_ratio, 
        include_zero_gene, eval_batch_size, device
    )
    
    # get unique class labels
    class_labels = np.unique(y_true)
    
    # plot ROC curves and get AUC values
    roc_auc, macro_auc = plot_multiclass_roc(y_probs, y_true, class_labels, id2type, save_dir)
    
    # plot AUC heatmap
    auc_df = plot_auc_heatmap(roc_auc, class_labels, id2type, save_dir)
    
    # save the AUC results
    results = {
        'class_auc': roc_auc,
        'macro_auc': macro_auc,
        'auc_dataframe': auc_df
    }
    
    with open(save_dir / "auc_results.pkl", "wb") as f:
        pickle.dump(results, f)
    
    print(f"Macro-average AUC: {macro_auc:.4f}")
    print(f"Results saved to {save_dir}")
    
    return results

## Ablation

In [ ]:
# HeadZeroer for scGPT
class HeadZeroer(torch.nn.Module):
    """A module that zeroes out a specific head's contribution"""
    def __init__(self, head_idx, head_dim):
        super().__init__()
        self.head_idx = head_idx
        self.head_dim = head_dim
        self.start_idx = head_idx * head_dim
        self.end_idx = (head_idx + 1) * head_dim
        print(f"Initializing HeadZeroer for head {head_idx} (indices {self.start_idx}-{self.end_idx})")
        
    def forward(self, hidden_states):
        """
        Zero out the specific head's contribution in the hidden states.
        hidden_states: [batch_size, seq_len, hidden_size]
        """
        # Create a copy to avoid modifying the original
        modified = hidden_states.clone()
        
        # Zero out the part corresponding to this head
        modified[:, :, self.start_idx:self.end_idx] = 0.0
        
        return modified


class SeqDataset(Dataset):
    def __init__(self, data: dict):
        self.data = data

    def __len__(self):
        return self.data["gene_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.data.items()}

In [ ]:
def apply_head_zeroing(model, layer_idx, head_idx):
    """
    Apply head zeroing to a specific attention head by inserting a HeadZeroer module
    after the attention output in the specified layer.
    """
    print(f"Applying head zeroing to layer {layer_idx}, head {head_idx}")
    
    # Ensure we're working with the correct structure
    if not hasattr(model, 'transformer_encoder') or not hasattr(model.transformer_encoder, 'layers'):
        raise AttributeError("Model doesn't have the expected transformer_encoder.layers structure")
        
    target_layer = model.transformer_encoder.layers[layer_idx]
    
    # Get dimensions
    if hasattr(model, 'nhead'):
        num_heads = model.nhead
        print(f"num_heads: {num_heads}")
    else:
        num_heads = 8
        print(f"Using configured num_heads: {num_heads}")
    
    if hasattr(model, 'embsize'):
        hidden_size = model.embsize
        print(f"hidden_size: {hidden_size}")
    else:
        hidden_size = 512
        print(f"Using configured hidden_size: {hidden_size}")
    
    # Calculate head dimension
    head_dim = hidden_size // num_heads
    
    # Capture the original attention output module
    if hasattr(target_layer.self_attn, 'out_proj'):
        original_attention_output = target_layer.self_attn.out_proj
    else:
        raise AttributeError(f"self_attn in layer {layer_idx} doesn't have out_proj attribute")
    
    # Create a HeadZeroer module
    head_zeroer = HeadZeroer(head_idx, head_dim)
    
    # Create a combined module that applies both the original processing and our zeroing
    class CombinedModule(torch.nn.Module):
        def __init__(self, original_module, zeroer):
            super().__init__()
            self.original_module = original_module
            self.zeroer = zeroer
            
        def forward(self, x):
            # First run the original module
            outputs = self.original_module(x)
            # Then apply our zeroer
            return self.zeroer(outputs)
    
    # Replace the original module with our combined one
    target_layer.self_attn.out_proj = CombinedModule(original_attention_output, head_zeroer)
    print(f"✓ Inserted HeadZeroer after attention output in layer {layer_idx} for head {head_idx}")
    
    return model

In [ ]:
# Apply head zeroing to multiple heads
def apply_multiple_head_zeroing(model, heads_to_ablate):
    """Apply head zeroing to multiple attention heads."""
    # Make a deep copy of the model to avoid modifying the original
    ablated_model = copy.deepcopy(model)
    
    # Check how many layers are available
    if hasattr(ablated_model, 'transformer_encoder') and hasattr(ablated_model.transformer_encoder, 'layers'):
        num_layers = len(ablated_model.transformer_encoder.layers)
        print(f"Model has {num_layers} transformer layers")
    else:
        print("WARNING: Could not determine number of layers")
        num_layers = 12  # Assuming 12 layers based on your parameter dump
    
    for head_info in heads_to_ablate:
        layer_idx = head_info["layer"]
        head_idx = head_info["head"]
        
        # Skip if layer index is out of bounds
        if layer_idx >= num_layers:
            print(f"Skipping Layer{layer_idx}-Head{head_idx}: Layer index out of bounds (max: {num_layers-1})")
            continue
            
        print(f"Ablating {head_info.get('name', f'Layer{layer_idx}-Head{head_idx}')}")
        try:
            ablated_model = apply_head_zeroing(ablated_model, layer_idx, head_idx)
        except Exception as e:
            print(f"Failed to ablate Layer{layer_idx}-Head{head_idx}: {e}")
            traceback.print_exc()
    
    return ablated_model

In [ ]:
# Evaluation function
def evaluate_model(model, data_loader, device, vocab, pad_token, num_types, input_batch_labels=False, config=None):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    all_results = []
    
    print("Evaluating model...")
    
    with torch.no_grad():
        for batch_data in data_loader:
            input_gene_ids = batch_data["gene_ids"].to(device)
            input_values = batch_data["values"].to(device)
            batch_labels = batch_data["batch_labels"].to(device)
            celltype_labels = batch_data["celltype_labels"].to(device)
            
            src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])
            
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                batch_labels=batch_labels if input_batch_labels or (config and config["DSBN"]) else None,
                CLS=True,
                CCE=False,
                MVC=False,
                ECS=False,
                do_sample=False,
            )
            
            outputs = output_dict["cls_output"]
            #check if outputs has logits by printing a small section
            #print(f"Outputs shape: {outputs.shape}")
            #print(f"Outputs sample: {outputs[0, :5]}")
            #print(f"Outputs sample: {outputs[1]}")
            # Get predicted probabilities and labels
            probs = F.softmax(outputs, dim=1).cpu().numpy()
            predictions = outputs.argmax(1).cpu().numpy()
            
            all_preds.extend(predictions)
            all_labels.extend(celltype_labels.cpu().numpy())
            all_probs.extend(probs)
            
            # Store results
            for i in range(len(input_gene_ids)):
                tokens = vocab.lookup_tokens(input_gene_ids[i].cpu().tolist())
                all_results.append({
                    'tokens': str(tokens),
                    'true_label': int(celltype_labels[i].cpu().numpy()),
                    'predicted': int(predictions[i]),
                    'prob_values': probs[i].tolist()
                })
    
    # Calculate accuracy
    accuracy = accuracy_score(all_labels, all_preds)
    
    # For multiclass, we calculate AUC for each class
    class_aucs = []
    for i in range(num_types):
        # Create binary labels for this class
        binary_labels = np.array(all_labels) == i
        if np.sum(binary_labels) > 0:  # Only calculate if we have positive examples
            # Get probabilities for this class
            class_probs = np.array([p[i] for p in all_probs])
            fpr, tpr, _ = roc_curve(binary_labels, class_probs)
            class_aucs.append(auc(fpr, tpr))
    
    # Average AUC across all classes
    mean_auc = np.mean(class_aucs) if class_aucs else 0
    return {
        'accuracy': accuracy,
        'mean_auc': mean_auc,
        'class_aucs': class_aucs,
        'labels': all_labels,
        'probs': all_probs,
        'detailed_results': all_results,
    }

In [ ]:
def plot_multi_class_roc(results_dict, output_path, model_name="baseline"):
    """Create a single ROC curve plot showing all classes for a specific model."""
    plt.figure(figsize=(12, 10))
    
    # Get data from the specified model
    labels = results_dict[model_name]['labels']
    probs = results_dict[model_name]['probs']
    
    # Get class names if available, otherwise use indices
    class_names = []
    try:
        if 'class_names' in results_dict:
            class_names = results_dict['class_names']
    except:
        pass
    
    # Use random colors for each class
    colors = plt.cm.get_cmap('tab20', len(probs[0]))
    
    # Get unique class labels
    class_labels = np.unique(labels)
    
    # Binarize labels for one-vs-rest calculation
    from sklearn.preprocessing import label_binarize
    y_true_bin = label_binarize(labels, classes=class_labels)
    
    # Calculate macro-average AUC using sklearn's function
    from sklearn.metrics import roc_auc_score
    macro_auc = roc_auc_score(y_true_bin, np.array(probs), multi_class='ovr', average='macro')
    
    # Plot ROC curve for each class
    for class_idx in range(len(probs[0])):
        # Create binary labels for this class
        binary_labels = np.array(labels) == class_idx
        
        # Get probabilities for this class
        class_probs = np.array([p[class_idx] for p in probs])
        
        # Only calculate if we have positive examples
        if np.sum(binary_labels) > 0:
            fpr, tpr, _ = roc_curve(binary_labels, class_probs)
            roc_auc = auc(fpr, tpr)
            
            # Use class name if available, otherwise use index
            class_label = f"Class {class_idx}"
            if class_idx < len(class_names):
                class_label = class_names[class_idx]
                
            plt.plot(
                fpr, tpr, 
                color=colors(class_idx), 
                label=f"{class_label} (AUC = {roc_auc:.2f})", 
                linewidth=2
            )
    
    # Add diagonal line (random classifier)
    plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.50)')
    
    # Add macro average to title
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.title(f'Multi-class ROC Curves (One-vs-Rest)\nMacro-Average AUC = {macro_auc:.4f}', fontsize=16)
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(True, alpha=0.3)
    
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Multi-class ROC curves saved to {output_path}")
    plt.close()
    
    # Return the macro-average AUC
    return macro_auc

In [ ]:
def plot_multi_class_roc(results_dict, output_path, model_name="baseline"):
    """Create a single ROC curve plot showing all classes for a specific model."""
    plt.figure(figsize=(12, 10))
    
    # Get data from the specified model
    labels = results_dict[model_name]['labels']
    probs = results_dict[model_name]['probs']
    
    # Get class names if available, otherwise use indices
    class_names = []
    try:
        if 'class_names' in results_dict:
            class_names = results_dict['class_names']
    except:
        pass
    
    # Use random colors for each class
    colors = plt.cm.get_cmap('tab20', len(probs[0]))
    
    # Get unique class labels
    class_labels = np.unique(labels)
    
    # Binarize labels for one-vs-rest calculation
    from sklearn.preprocessing import label_binarize
    y_true_bin = label_binarize(labels, classes=class_labels)
    
    # Calculate macro-average AUC using sklearn's function
    from sklearn.metrics import roc_auc_score
    macro_auc = roc_auc_score(y_true_bin, np.array(probs), multi_class='ovr', average='macro')
    
    # Plot ROC curve for each class
    for class_idx in range(len(probs[0])):
        # Create binary labels for this class
        binary_labels = np.array(labels) == class_idx
        
        # Get probabilities for this class
        class_probs = np.array([p[class_idx] for p in probs])
        
        # Only calculate if we have positive examples
        if np.sum(binary_labels) > 0:
            fpr, tpr, _ = roc_curve(binary_labels, class_probs)
            roc_auc = auc(fpr, tpr)
            
            # Use class name if available, otherwise use index
            class_label = f"Class {class_idx}"
            if class_idx < len(class_names):
                class_label = class_names[class_idx]
                
            plt.plot(
                fpr, tpr, 
                color=colors(class_idx), 
                label=f"{class_label} (AUC = {roc_auc:.2f})", 
                linewidth=2
            )
    
    # Add diagonal line (random classifier)
    plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.50)')
    
    # Add macro average to title
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.title(f'Multi-class ROC Curves (One-vs-Rest)\nMacro-Average AUC = {macro_auc:.4f}', fontsize=16)
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(True, alpha=0.3)
    
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Multi-class ROC curves saved to {output_path}")
    plt.close()
    
    # Return the macro-average AUC
    return macro_auc

In [ ]:
def compare_model_rocs(results_dict, output_path, important_head_title, unimportant_head_title):
    """Create separate multi-class ROC curve plots for each model."""
    # Create ROC curve for baseline model
    plot_multi_class_roc(results_dict, f"{output_path}/baseline_roc_curves.png", "baseline")
    
    # Create ROC curve for important heads model
    plot_multi_class_roc(results_dict, f"{output_path}/important_roc_curves.png", "important")
    
    # Create ROC curve for unimportant heads model
    plot_multi_class_roc(results_dict, f"{output_path}/unimportant_roc_curves.png", "unimportant")

In [ ]:
def plot_accuracy_comparison(results_dict, output_path, important_head_title, unimportant_head_title):
    """Plot bar chart comparing accuracy across models."""
    plt.figure(figsize=(10, 6))
    
    models = ['baseline', 'important', 'unimportant']
    labels = ['Baseline', important_head_title, unimportant_head_title]
    accuracies = [results_dict[m]['accuracy'] for m in models]
    aucs = [results_dict[m]['mean_auc'] for m in models]
    
    x = np.arange(len(models))
    width = 0.35
    
    plt.bar(x - width/2, accuracies, width, label='Accuracy')
    plt.bar(x + width/2, aucs, width, label='Mean AUC')
    
    plt.ylabel('Score')
    plt.title('Model Performance Comparison')
    plt.xticks(x, labels)
    plt.legend()
    
    # Add values on top of bars
    for i, v in enumerate(accuracies):
        plt.text(i - width/2, v + 0.01, f'{v:.3f}', ha='center')
    for i, v in enumerate(aucs):
        plt.text(i + width/2, v + 0.01, f'{v:.3f}', ha='center')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Accuracy comparison saved to {output_path}")
    plt.close()

In [ ]:
results_dir = "" # path to ablation results dir

if dataset == "ms":
    heads_file = "" # path to heads file
elif dataset == "pancreas":
    heads_file = "" # path to heads file

In [ ]:
def load_heads_from_file(file_path):
    """
    Dynamically load important_heads and unimportant_heads from a Python file.
    """
    # Create empty namespace to execute the file in
    namespace = {}
    
    # Read and execute the file content
    with open(file_path, 'r') as f:
        exec(f.read(), namespace)
    
    # Extract the head lists from the namespace
    important_heads = namespace.get('important_heads', [])
    unimportant_heads = namespace.get('unimportant_heads', [])
    
    return important_heads, unimportant_heads

In [ ]:
# Load both types of heads from the same file
important_heads, unimportant_heads = load_heads_from_file(heads_file)

print(f"Loaded {len(important_heads)} important heads and {len(unimportant_heads)} unimportant heads")

In [ ]:
adata = adata_test
model = best_model

In [ ]:
# Process data for testing
all_counts = (
    adata.layers[input_layer_key].A
    if issparse(adata.layers[input_layer_key])
    else adata.layers[input_layer_key]
)

celltypes_labels = adata.obs["celltype_id"].tolist()  # make sure count from 0
celltypes_labels = np.array(celltypes_labels[:499])

batch_ids = adata.obs["batch_id"].tolist()
batch_ids = np.array(batch_ids[:499])

tokenized_test = tokenize_and_pad_batch(
    all_counts[:499],
    gene_ids,
    max_len=max_seq_len,
    vocab=vocab,
    pad_token=pad_token,
    pad_value=pad_value,
    append_cls=True,  # append <cls> token at the beginning
    include_zero_gene=include_zero_gene,
)

input_values_test = random_mask_value(
    tokenized_test["values"],
    mask_ratio=mask_ratio,
    mask_value=mask_value,
    pad_value=pad_value,
)

test_data_pt = {
    "gene_ids": tokenized_test["genes"],
    "values": input_values_test,
    "target_values": tokenized_test["values"],
    "batch_labels": torch.from_numpy(batch_ids).long(),
    "celltype_labels": torch.from_numpy(celltypes_labels).long(),
}

test_loader = DataLoader(
    dataset=SeqDataset(test_data_pt),
    batch_size=eval_batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=min(len(os.sched_getaffinity(0)), eval_batch_size // 2),
    pin_memory=True,
)

In [ ]:
def plot_performance_by_percentage(results_dict, percentages, output_path, important_head_title, unimportant_head_title):
    """
    Create a line plot showing how accuracy and AUC change with increasing percentage of heads ablated.
    """
    plt.figure(figsize=(12, 8))
    
    # Get baseline values
    baseline_acc = results_dict['baseline']['accuracy']
    baseline_auc = results_dict['baseline']['mean_auc']
    
    # Get values for each percentage of important heads
    important_accs = [results_dict[f'important_{p}']['accuracy'] for p in percentages]
    important_aucs = [results_dict[f'important_{p}']['mean_auc'] for p in percentages]
    
    # Get values for each percentage of unimportant heads
    unimportant_accs = [results_dict[f'unimportant_{p}']['accuracy'] for p in percentages]
    unimportant_aucs = [results_dict[f'unimportant_{p}']['mean_auc'] for p in percentages]
    
    # Create subplot for AUC
    plt.subplot(2, 1, 1)
    
    # Plot AUC lines
    plt.plot(percentages, important_aucs, 'o-', color='red', linewidth=2, 
             label=f'{important_head_title} Ablated')
    plt.plot(percentages, unimportant_aucs, 'o-', color='green', linewidth=2, 
             label=f'{unimportant_head_title} Ablated')
    
    # Add a horizontal line for the baseline
    plt.axhline(y=baseline_auc, color='blue', linestyle='--', 
                label=f'Baseline (AUC = {baseline_auc:.3f})')
    
    # Add labels and legend
    plt.ylabel('Mean AUC')
    plt.title('Impact of Ablation on AUC Performance')
    plt.grid(True, alpha=0.3)
    plt.legend(loc='best')
    
    # Make the x-axis show percentages
    plt.xticks(percentages)
    
    # Create subplot for Accuracy
    plt.subplot(2, 1, 2)
    
    # Plot Accuracy lines
    plt.plot(percentages, important_accs, 'o-', color='red', linewidth=2, 
             label=f'{important_head_title} Ablated')
    plt.plot(percentages, unimportant_accs, 'o-', color='green', linewidth=2, 
             label=f'{unimportant_head_title} Ablated')
    
    # Add a horizontal line for the baseline
    plt.axhline(y=baseline_acc, color='blue', linestyle='--', 
                label=f'Baseline (Acc = {baseline_acc:.3f})')
    
    # Add labels and legend
    plt.xlabel('Percentage of Heads Ablated (%)')
    plt.ylabel('Accuracy')
    plt.title('Impact of Ablation on Accuracy')
    plt.grid(True, alpha=0.3)
    plt.legend(loc='best')
    
    # Make the x-axis show percentages
    plt.xticks(percentages)
    
    # Adjust layout and save
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Performance by percentage plot saved to {output_path}")
    plt.close()

In [ ]:
def plot_all_roc_curves(results_dict, percentages, output_path, important_head_title, unimportant_head_title):
    """
    Create ROC curves comparing baseline and all percentages of important and unimportant heads.
    This uses one-vs-rest approach for multi-class classification.
    """
    print(f"DEBUG: Starting plot_all_roc_curves function")
    print(f"DEBUG: Output path: {output_path}")
    print(f"DEBUG: Percentages to plot: {percentages}")
    print(f"DEBUG: Important head title: {important_head_title}")
    print(f"DEBUG: Unimportant head title: {unimportant_head_title}")
    print(f"DEBUG: Available keys in results_dict: {list(results_dict.keys())}")
    
    plt.figure(figsize=(15, 10))
    
    # Get baseline data
    baseline_labels = np.array(results_dict['baseline']['labels'])
    baseline_probs = np.array(results_dict['baseline']['probs'])
    
    print(f"DEBUG: Baseline labels shape: {baseline_labels.shape}")
    print(f"DEBUG: Baseline probs shape: {baseline_probs.shape}")
    
    # Determine the number of classes from the predictions shape
    num_classes = baseline_probs.shape[1]
    print(f"DEBUG: Number of classes detected: {num_classes}")
    
    # Get unique class labels from the actual labels
    unique_labels = np.unique(baseline_labels)
    n_unique = len(unique_labels)
    print(f"DEBUG: Unique classes in labels: {unique_labels}")
    print(f"DEBUG: Number of unique classes: {n_unique}")
    
    if n_unique != num_classes:
        print(f"WARNING: Number of unique classes in labels ({n_unique}) doesn't match prediction columns ({num_classes})")
    
    # Plot random classifier
    plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.500)')
    
    # Calculate AUC for baseline
    binary_aucs = []
    
    # Store the macro-average ROC curves for each configuration
    baseline_tprs = []
    baseline_fprs = []
    
    print(f"DEBUG: Calculating ROC curves for baseline (each class)")
    # Create a custom binarized matrix that matches the shape of the predictions
    for class_idx in range(num_classes):
        binary_labels = (baseline_labels == class_idx).astype(int)
        class_probs = baseline_probs[:, class_idx]
        
        print(f"DEBUG: Class {class_idx}: Positive examples: {np.sum(binary_labels)}, Negative examples: {len(binary_labels) - np.sum(binary_labels)}")
        
        # Skip classes with no positive examples
        if np.sum(binary_labels) == 0:
            print(f"DEBUG: Skipping class {class_idx} - no positive examples")
            continue
            
        # Calculate ROC and AUC
        try:
            fpr, tpr, _ = roc_curve(binary_labels, class_probs)
            baseline_fprs.append(fpr)
            baseline_tprs.append(tpr)
            class_auc = auc(fpr, tpr)
            binary_aucs.append(class_auc)
            
            print(f"DEBUG: Class {class_idx} - AUC: {class_auc:.4f}")
            
            # Plot baseline curve for this class (transparent)
            plt.plot(fpr, tpr, color='blue', alpha=0.2, linewidth=1)
        except Exception as e:
            print(f"ERROR: Error calculating ROC for class {class_idx}: {e}")
            continue
    
    # Calculate macro-average AUC for baseline
    baseline_macro_auc = np.mean(binary_aucs) if binary_aucs else 0
    print(f"DEBUG: Baseline macro-average AUC: {baseline_macro_auc:.4f} (from {len(binary_aucs)} classes)")
    
    # Calculate mean ROC curve for baseline
    mean_fpr = np.linspace(0, 1, 100)
    mean_tprs = []
    
    for fpr, tpr in zip(baseline_fprs, baseline_tprs):
        mean_tprs.append(np.interp(mean_fpr, fpr, tpr))
    
    mean_tpr = np.mean(mean_tprs, axis=0)
    
    # Plot the mean ROC curve for baseline
    plt.plot(
        mean_fpr, mean_tpr,
        color='blue', linewidth=3, 
        label=f'Baseline (Macro AUC = {baseline_macro_auc:.3f})'
    )
    
    # Colors for important heads (shades of red)
    important_colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(percentages)))
    
    # Colors for unimportant heads (shades of green)
    unimportant_colors = plt.cm.Greens(np.linspace(0.3, 0.9, len(percentages)))
    
    print(f"DEBUG: Processing important heads")
    # Plot important heads at each percentage
    for i, percentage in enumerate(percentages):
        key = f'important_{percentage}'
        print(f"DEBUG: Processing {key}")
        
        if key not in results_dict:
            print(f"DEBUG: Key {key} not found in results_dict, skipping")
            continue
            
        labels = np.array(results_dict[key]['labels'])
        probs = np.array(results_dict[key]['probs'])
        
        print(f"DEBUG: {key} - labels shape: {labels.shape}, probs shape: {probs.shape}")
        
        # Calculate AUCs for each class
        imp_binary_aucs = []
        imp_fprs = []
        imp_tprs = []
        
        for class_idx in range(num_classes):
            binary_labels = (labels == class_idx).astype(int)
            class_probs = probs[:, class_idx]
            
            positive_count = np.sum(binary_labels)
            print(f"DEBUG: {key} - Class {class_idx}: Positive examples: {positive_count}")
            
            # Skip classes with no positive examples
            if positive_count == 0:
                print(f"DEBUG: {key} - Skipping class {class_idx} - no positive examples")
                continue
                
            try:
                fpr, tpr, _ = roc_curve(binary_labels, class_probs)
                imp_fprs.append(fpr)
                imp_tprs.append(tpr)
                class_auc = auc(fpr, tpr)
                imp_binary_aucs.append(class_auc)
                print(f"DEBUG: {key} - Class {class_idx} - AUC: {class_auc:.4f}")
            except Exception as e:
                print(f"ERROR: Error calculating ROC for {key}, class {class_idx}: {e}")
                continue
        
        # Calculate macro-average AUC
        macro_auc = np.mean(imp_binary_aucs) if imp_binary_aucs else 0
        print(f"DEBUG: {key} - Macro-average AUC: {macro_auc:.4f} (from {len(imp_binary_aucs)} classes)")
        
        # Calculate mean ROC curve
        mean_imp_tprs = []
        for fpr, tpr in zip(imp_fprs, imp_tprs):
            mean_imp_tprs.append(np.interp(mean_fpr, fpr, tpr))
        
        mean_imp_tpr = np.mean(mean_imp_tprs, axis=0) if mean_imp_tprs else np.zeros_like(mean_fpr)
        
        # Plot the actual curve
        plt.plot(
            mean_fpr, mean_imp_tpr,
            color=important_colors[i], linewidth=2, 
            label=f'{important_head_title} {percentage}% (Macro AUC = {macro_auc:.3f})'
        )
    
    print(f"DEBUG: Processing unimportant heads")
    # Plot unimportant heads at each percentage
    for i, percentage in enumerate(percentages):
        key = f'unimportant_{percentage}'
        print(f"DEBUG: Processing {key}")
        
        if key not in results_dict:
            print(f"DEBUG: Key {key} not found in results_dict, skipping")
            continue
            
        labels = np.array(results_dict[key]['labels'])
        probs = np.array(results_dict[key]['probs'])
        
        print(f"DEBUG: {key} - labels shape: {labels.shape}, probs shape: {probs.shape}")
        
        # Calculate AUCs for each class
        unimp_binary_aucs = []
        unimp_fprs = []
        unimp_tprs = []
        
        for class_idx in range(num_classes):
            binary_labels = (labels == class_idx).astype(int)
            class_probs = probs[:, class_idx]
            
            positive_count = np.sum(binary_labels)
            print(f"DEBUG: {key} - Class {class_idx}: Positive examples: {positive_count}")
            
            # Skip classes with no positive examples
            if positive_count == 0:
                print(f"DEBUG: {key} - Skipping class {class_idx} - no positive examples")
                continue
                
            try:
                fpr, tpr, _ = roc_curve(binary_labels, class_probs)
                unimp_fprs.append(fpr)
                unimp_tprs.append(tpr)
                class_auc = auc(fpr, tpr)
                unimp_binary_aucs.append(class_auc)
                print(f"DEBUG: {key} - Class {class_idx} - AUC: {class_auc:.4f}")
            except Exception as e:
                print(f"ERROR: Error calculating ROC for {key}, class {class_idx}: {e}")
                continue
        
        # Calculate macro-average AUC
        macro_auc = np.mean(unimp_binary_aucs) if unimp_binary_aucs else 0
        print(f"DEBUG: {key} - Macro-average AUC: {macro_auc:.4f} (from {len(unimp_binary_aucs)} classes)")
        
        # Calculate mean ROC curve
        mean_unimp_tprs = []
        for fpr, tpr in zip(unimp_fprs, unimp_tprs):
            mean_unimp_tprs.append(np.interp(mean_fpr, fpr, tpr))
        
        mean_unimp_tpr = np.mean(mean_unimp_tprs, axis=0) if mean_unimp_tprs else np.zeros_like(mean_fpr)
        
        # Plot the actual curve
        plt.plot(
            mean_fpr, mean_unimp_tpr,
            color=unimportant_colors[i], linewidth=2, linestyle='--',
            label=f'{unimportant_head_title} {percentage}% (Macro AUC = {macro_auc:.3f})'
        )
    
    print(f"DEBUG: Finalizing plot")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'ROC Curves for Different Percentages of Ablated Heads\n(Macro-Average)', fontsize=16)
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(True, alpha=0.3)
    
    print(f"DEBUG: Saving plot to {output_path}")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"SUCCESS: All ROC curves saved to {output_path}")
    plt.close()
    print(f"DEBUG: Function completed successfully")

In [ ]:
percentages = "5,10,20,30,40,50"
# Parse percentages from arguments
percentages = [int(p) for p in percentages.split(',')]

In [ ]:
# Dictionary to store all results for comparison
results_dict = {}

# Define the important and unimportant head titles based on experiment name
if "pancreas_ductal" in experiment_name.lower():
    important_head_title = 'Pancreas Ductal Cell Important Heads'
    unimportant_head_title = 'Pancreas Ductal Cell Unimportant Heads'
elif "GOBP_cell_dev" in experiment_name.lower():
    important_head_title = 'GOBP Cell Development Important Heads'
    unimportant_head_title = 'GOBP Cell Development Unimportant Heads'
else:
    important_head_title = 'Important Heads'
    unimportant_head_title = 'Unimportant Heads'

In [ ]:
# Print class distribution in test dataset
logger.info("\n====== Class Distribution in Datasets ======")

# For test data from test_loader
test_labels = []
for batch in test_loader:
    test_labels.extend(batch["celltype_labels"].numpy())

unique_test_labels, test_counts = np.unique(test_labels, return_counts=True)
test_percentages = test_counts / len(test_labels) * 100

logger.info("Test set class distribution:")
for label, count, percentage in zip(unique_test_labels, test_counts, test_percentages):
    logger.info(f"  Class {label} ({id2type[label]}): {count} samples ({percentage:.2f}%)")

In [ ]:
# Get total number of heads in the model
num_layers = nlayers
num_heads = nhead
total_heads = num_layers * num_heads
logger.info(f"Model has {total_heads} total attention heads ({num_layers} layers × {num_heads} heads)")

# Calculate number of heads for each percentage
heads_per_percentage = {p: int(round(p * total_heads / 100)) for p in percentages}
logger.info(f"Will ablate these numbers of heads: {heads_per_percentage}")

In [ ]:
# Dictionary to store all results for comparison
results_dict = {}

# 1. Evaluate the original model (baseline)
logger.info("\n====== Evaluating Original Model (All Heads Intact) ======")

baseline_results = evaluate_model(
    model, 
    test_loader, 
    device, 
    vocab, 
    pad_token, 
    num_types,
    INPUT_BATCH_LABELS,
    config
)
results_dict['baseline'] = baseline_results

# Save baseline detailed results
baseline_df = pd.DataFrame(baseline_results['detailed_results'])
os.makedirs(results_dir, exist_ok=True)
baseline_df.to_csv(f"{results_dir}/{experiment_name}_baseline_results.tsv", sep='\t', index=False)

logger.info(f"Baseline model - Accuracy: {baseline_results['accuracy']:.4f}, Mean AUC: {baseline_results['mean_auc']:.4f}")

In [ ]:
# 2. Ablate important heads at each percentage
for percentage in percentages:
    num_heads_to_ablate = heads_per_percentage[percentage]
    
    # Take the first N heads from the important heads list
    heads_to_ablate = important_heads[:num_heads_to_ablate]
    
    logger.info(f"\n====== Ablating Top {percentage}% Important Heads ({num_heads_to_ablate} heads) ======")
    head_names = [head.get("name", f"Layer{head['layer']}-Head{head['head']}") for head in heads_to_ablate]
    logger.info(f"Ablating heads: {', '.join(head_names)}")
    
    important_model = apply_multiple_head_zeroing(model, heads_to_ablate)
    important_model.to(device)
    
    results = evaluate_model(
        important_model, 
        test_loader, 
        device, 
        vocab, 
        pad_token, 
        num_types,
        INPUT_BATCH_LABELS,
        config
    )
    results_dict[f'important_{percentage}'] = results
    
    # Save detailed results
    df = pd.DataFrame(results['detailed_results'])
    df.to_csv(f"{results_dir}/{experiment_name}_important_heads_{percentage}percent_results.tsv", sep='\t', index=False)
    
    logger.info(f"{important_head_title} {percentage}% ablated - Accuracy: {results['accuracy']:.4f}, Mean AUC: {results['mean_auc']:.4f}")
    
    # Clean up to save memory
    del important_model
    torch.cuda.empty_cache()

In [ ]:
# 3. Ablate unimportant heads at each percentage
for percentage in percentages:
    num_heads_to_ablate = heads_per_percentage[percentage]
    
    # Take the first N heads from the unimportant heads list
    heads_to_ablate = unimportant_heads[:num_heads_to_ablate]
    
    logger.info(f"\n====== Ablating Top {percentage}% Unimportant Heads ({num_heads_to_ablate} heads) ======")
    head_names = [head.get("name", f"Layer{head['layer']}-Head{head['head']}") for head in heads_to_ablate]
    logger.info(f"Ablating heads: {', '.join(head_names)}")
    
    unimportant_model = apply_multiple_head_zeroing(model, heads_to_ablate)
    unimportant_model.to(device)
    
    results = evaluate_model(
        unimportant_model, 
        test_loader, 
        device, 
        vocab, 
        pad_token, 
        num_types,
        INPUT_BATCH_LABELS,
        config
    )
    results_dict[f'unimportant_{percentage}'] = results
    
    # Save detailed results
    df = pd.DataFrame(results['detailed_results'])
    df.to_csv(f"{results_dir}/{experiment_name}_unimportant_heads_{percentage}percent_results.tsv", sep='\t', index=False)
    
    logger.info(f"{unimportant_head_title} {percentage}% ablated - Accuracy: {results['accuracy']:.4f}, Mean AUC: {results['mean_auc']:.4f}")
    
    # Clean up to save memory
    del unimportant_model
    torch.cuda.empty_cache()

In [ ]:
# 4. Create summary table
summary_data = {
    'Model': ['Baseline'],
    'Accuracy': [results_dict['baseline']['accuracy']],
    'Mean AUC': [results_dict['baseline']['mean_auc']]
}

for percentage in percentages:
    summary_data['Model'].append(f'{important_head_title} {percentage}% Ablated')
    summary_data['Accuracy'].append(results_dict[f'important_{percentage}']['accuracy'])
    summary_data['Mean AUC'].append(results_dict[f'important_{percentage}']['mean_auc'])
    
for percentage in percentages:
    summary_data['Model'].append(f'{unimportant_head_title} {percentage}% Ablated')
    summary_data['Accuracy'].append(results_dict[f'unimportant_{percentage}']['accuracy'])
    summary_data['Mean AUC'].append(results_dict[f'unimportant_{percentage}']['mean_auc'])

summary_df = pd.DataFrame(summary_data)
summary_path = f"{results_dir}/{experiment_name}_ablation_summary.tsv"
summary_df.to_csv(summary_path, sep='\t', index=False)

# Print summary table
logger.info("\n====== ABLATION EXPERIMENT SUMMARY ======")
logger.info(summary_df.to_string())

In [ ]:
# 5. Create visualizations
logger.info("\n====== Creating Visualizations ======")

# plot performance across percentages
plot_performance_by_percentage(
    results_dict, 
    percentages, 
    f"{results_dir}/{experiment_name}_performance_by_percentage.png", 
    important_head_title, 
    unimportant_head_title
)

# Plot all ROC curves
plot_all_roc_curves(
    results_dict, 
    percentages, 
    f"{results_dir}/{experiment_name}_all_roc_curves.png", 
    important_head_title, 
    unimportant_head_title
)

# Save the full results dictionary for further analysis
with open(f"{results_dir}/{experiment_name}_full_results.pkl", 'wb') as f:
    pickle.dump(results_dict, f)

logger.info(f"\nExperiment complete! Results saved to {results_dir}")

## Attention

In [ ]:
def analyze_attention_heads(model, adata, device, vocab, nlayers, nhead, pad_token, config):
    """
    Analyze attention heads directly from AnnData object
    
    Args:
        model: The model to analyze
        adata: AnnData object containing data
        device: Torch device
        vocab: Vocabulary
        nlayers: Number of layers in model
        nhead: Number of attention heads per layer
        pad_token: Token to use for padding
        config: Configuration object
        
    Returns:
        Dictionary with attention scores for each layer and head
    """
    # Process data for testing
    all_counts = (
        adata.layers[input_layer_key].A
        if issparse(adata.layers[input_layer_key])
        else adata.layers[input_layer_key]
    )

    celltypes_labels = adata.obs["celltype_id"].tolist()  # make sure count from 0
    celltypes_labels = np.array(celltypes_labels[:499])

    batch_ids = adata.obs["batch_id"].tolist()
    batch_ids = np.array(batch_ids[:499])

    tokenized_test = tokenize_and_pad_batch(
        all_counts[:499],
        gene_ids,
        max_len=max_seq_len,
        vocab=vocab,
        pad_token=pad_token,
        pad_value=pad_value,
        append_cls=True,  # append <cls> token at the beginning
        include_zero_gene=include_zero_gene,
    )

    input_values_test = random_mask_value(
        tokenized_test["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )

    test_data_pt = {
        "gene_ids": tokenized_test["genes"],
        "values": input_values_test,
        "target_values": tokenized_test["values"],
        "batch_labels": torch.from_numpy(batch_ids).long(),
        "celltype_labels": torch.from_numpy(celltypes_labels).long(),
    }

    test_loader = DataLoader(
        dataset=SeqDataset(test_data_pt),
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=min(len(os.sched_getaffinity(0)), eval_batch_size // 2),
        pin_memory=True,
    )

    # settings for prediction
    MLM = False  # whether to use masked language modeling, currently it is always on.
    CLS = True  # celltype classification objective
    ADV = False  # Adversarial training for batch correction
    CCE = False  # Contrastive cell embedding objective
    MVC = config.MVC  # Masked value prediction for cell embedding
    ECS = config.ecs_thres > 0  # Elastic cell similarity objective
    DAB = False  # Domain adaptation by reverse backpropagation, set to 2 for separate optimizer
    INPUT_BATCH_LABELS = False  # TODO: have these help MLM and MVC, while not to classifier
    input_emb_style = "continuous"  # "category" or "continuous" or "scaling"
    cell_emb_style = "cls"  # "avg-pool" or "w-pool" or "cls"
    adv_E_delay_epochs = 0  # delay adversarial training on encoder for a few epochs
    adv_D_delay_epochs = 0
    mvc_decoder_style = "inner product"
    ecs_threshold = config.ecs_thres
    dab_weight = config.dab_weight

    explicit_zero_prob = MLM and include_zero_gene  # whether explicit bernoulli for zeros
    do_sample_in_train = False and explicit_zero_prob  # sample the bernoulli in training

    num_layers = nlayers 
    num_heads = nhead  

    model.to(device)
    model.eval()

    # Log analysis info
    logger.info("\n====== Analyzing Attention Heads ======")
    logger.info(f"Model has {num_layers} layers with {num_heads} heads each")
    
    # Print class distribution in dataset
    logger.info("\n====== Class Distribution in Dataset ======")
    test_labels = []
    for batch in test_loader:
        test_labels.extend(batch["celltype_labels"].numpy())

    unique_test_labels, test_counts = np.unique(test_labels, return_counts=True)
    test_percentages = test_counts / len(test_labels) * 100

    logger.info("Dataset class distribution:")
    for label, count, percentage in zip(unique_test_labels, test_counts, test_percentages):
        logger.info(f"  Class {label} ({id2type[label]}): {count} samples ({percentage:.2f}%)")

    # Dictionary to store all examples and their scores for each head
    examples_scores_attention = {layer: {head: [] for head in range(num_heads)} for layer in range(num_layers)}

    for batch_num, batch_data in enumerate(test_loader):
        input_gene_ids = batch_data["gene_ids"].to(device)
        input_values = batch_data["values"].to(device)
        target_values = batch_data["target_values"].to(device)
        batch_labels = batch_data["batch_labels"].to(device)
        celltype_labels = batch_data["celltype_labels"].to(device)
        src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])

        # Output dictionary 
        output_dict = model(
            input_gene_ids,
            input_values,
            src_key_padding_mask=src_key_padding_mask,
            batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
            CLS=CLS,
            CCE=False,
            MVC=False,
            ECS=False,
            do_sample=do_sample_in_train,
        )
        
        input_tokens = [vocab.lookup_tokens(ids.tolist()) for ids in input_gene_ids]
        outputs = output_dict["cls_output"]
       
        # Get attention scores
        all_attentions = output_dict["attentions"]  # assuming the model was set with output_attentions=True
        # Print shape of attention outputs
        print(f"all_attentions contains {len(all_attentions)} layers")
        
        # Get labels
        batch_labels_list = celltype_labels.detach().cpu().numpy().tolist()
        
        # Get expression values
        input_values_list = input_values.detach().cpu().numpy().tolist()
        
        if batch_num == 0:
            logger.info(f"Processing {len(input_values_list)} examples per batch")

        # For each layer and head...
        for layer in range(num_layers):
            # Check shape of attention for this layer
            print(f"Layer {layer} attention shape: {all_attentions[layer].shape}")
            
            for head in range(num_heads):
                attention_scores = all_attentions[layer][:, head, :, :]
                # check shape per head
                print(f"Layer {layer}, Head {head}, Attention shape: {attention_scores.shape}")

                # Tokens, attention matrices, labels, and input_values together
                for i, (tokens, att_matrix, label, values) in enumerate(zip(input_tokens, attention_scores, batch_labels_list, input_values_list)):
                    # Check indiividual attention matrix shape
                    if i == 0:  # Just print for the first item to avoid flooding output
                        print(f"Sample 0, Layer {layer}, Head {head}, Matrix shape: {att_matrix.shape}")
                        print(f"Min: {att_matrix.min().item()}, Max: {att_matrix.max().item()}, Mean: {att_matrix.mean().item()}")
                        
                    max_att_scores = att_matrix.max(dim=0)[0].detach().cpu().numpy()
                    # Append a tuple with max attention scores, tokens, label, and the specific input_values
                    examples_scores_attention[layer][head].append((max_att_scores, tokens, label, values))
    
    # Log completion
    total_examples = len(examples_scores_attention[0][0])
    logger.info(f"\nAttention analysis complete! Processed {total_examples} total examples")
    
    return examples_scores_attention

In [ ]:
results = analyze_attention_heads(model, 
                                  adata_test, 
                                  device, 
                                  vocab, 
                                  nlayers,
                                  nhead, 
                                  pad_token, 
                                  config)

In [ ]:
# unpack results
examples_scores_attention  = results

# directory exists
os.makedirs(f'{full_path}/attention/{dataset_name}/{task_name}/', exist_ok=True)
#save scores in layer-indexed-files
for layer in range(nlayers):
    #examples_scores_attention for the layer
    attention_filename = f'{full_path}/attention/{dataset_name}/{task_name}/examples_scores_attention_layer{layer}.p'
    with open(attention_filename, 'wb') as f:
        pickle.dump(examples_scores_attention[layer], f)
    print(f'Attentions saved to {attention_filename}')

### Attention Heatmap

In [ ]:
# Define the path to the directory containing the pickle files
base = "" # path to attentions directory
data_dir = os.path.join(base, dataset, task_name)

pkl_files = [
    'examples_scores_attention_layer0.p',
    'examples_scores_attention_layer1.p',
    'examples_scores_attention_layer2.p',
    'examples_scores_attention_layer3.p',
    'examples_scores_attention_layer4.p',
    'examples_scores_attention_layer5.p',
    'examples_scores_attention_layer6.p',
    'examples_scores_attention_layer7.p',
    'examples_scores_attention_layer8.p',
    'examples_scores_attention_layer9.p',
    'examples_scores_attention_layer10.p',
    'examples_scores_attention_layer11.p',
]

In [ ]:
# Load all pickle files into memory
layers_data = []
for file_name in pkl_files:
    file_path = os.path.join(data_dir, file_name)
    with open(file_path, 'rb') as f:
        layers_data.append(pickle.load(f))

# Assuming all layers have the same number of heads and all heads have the same number of cells
num_layers = len(layers_data)
num_heads = len(layers_data[0])
num_cells = len(layers_data[0][0])
num_genes = len(layers_data[0][0][0][0]) - 1  # Assuming each cell contains data for the same number of genes

print('Check Data:', num_layers, num_heads, num_cells, num_genes)

In [ ]:
mean_score_df:pd.DataFrame = pd.DataFrame()

# Open the layer
for layer in range(12):
    # print("LAYER:", layer)
    with open(f'{data_dir}/examples_scores_attention_layer{layer}.p', 'rb') as f:
        results:dict = pickle.load(f)

    # Calculate mean score per sequence per head
    tmp_dict:dict = {}
    for head in results:
        # print("HEAD:", head)
        tmp_list:list = []
        for i in range(len(results[head])):
          # sequence = (results[head][i][0]).cpu().numpy()
          # modified_sequence = sequence[1:-1]  # Exclude the first and last token
          # tmp_list.append(np.mean(modified_sequence))
          tmp_list.append(np.mean((results[head][i][0])))
        tmp_dict[head] = tmp_list

    # Merge layer scores
    tmp_df:pd.DataFrame = pd.DataFrame(tmp_dict)
    tmp_df.columns = [f'head{i}' for i in range(8)]
    tmp_df['layer'] = f'layer{layer}'
    mean_score_df = pd.concat([mean_score_df, tmp_df])

# Save the results as a CSV
mean_score_df.to_csv(f'{data_dir}/examples_mean_{approach}_scores.csv', index=False)

del(tmp_df, tmp_dict, tmp_list, results, head, layer, f, i)

In [ ]:
# Calculate mean score for each layer
tmp_mean_df = mean_score_df.groupby('layer').mean()

# Normalize the scores within each layer
tmp_mean_df = tmp_mean_df.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=1)

# Sort layers
tmp_mean_df = tmp_mean_df.reindex(sorted(tmp_mean_df.index, key=lambda x: int(x[5:])))

# Plotting
plt.figure(1, figsize=(9, 6))
sns.set(color_codes=True)
sns.set(font_scale=0.9)
ax = sns.heatmap(tmp_mean_df, cmap='GnBu', cbar_kws={'label': 'Scale'})
ax.set(ylabel="Layers", xlabel="Heads")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=45)

plt.savefig(f"{data_dir}/scgpt_mean_attention_heatmap.png", dpi=300, bbox_inches='tight')

plt.show()

tmp_mean_df.to_csv(f'{data_dir}/tmp_mean_{approach}_scores.csv', index=False)

# Clean up
del(tmp_mean_df, ax)